# 📉 09 · Classification II: logistic regression, generative vs discriminative
### Model P(Y | X) directly: the logistic function, a linear decision boundary, the conditional log-likelihood and its gradient derived step by step, gradient ascent, MAP regularisation, and a real leukaemia diagnosis

**BiologyToML** · Part B — Supervised learning · Module 09 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/09_logistic_regression.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib, scikit-learn (data sets and checks only) |
| **Prerequisites** | [Module 04](04_calculus_and_optimization.ipynb) (derivatives, chain rule, gradient ascent), [Module 05](05_likelihood_mle_map.ipynb) (MLE, MAP) and [Module 08](08_knn_naive_bayes.ipynb) (classification, naïve Bayes) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `004_classification-LR` slides 1–37 — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **Real data:** the Golub et al. (1999) leukaemia expression data with the original 38/34 train/test split
> (genes are selected on the training patients only; the notebook prints the data source and falls back to a
> labelled simulated stand-in offline), Fisher's iris flowers (scikit-learn) and the Wisconsin
> diagnostic breast-cancer data (scikit-learn). **Simulated data (labelled SIMULATED):** the 1-D biomarker toy,
> the single-protein data set used to derive and visualise the likelihood, and the ring-shaped data of
> Section 9.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why linear regression is a poor model for 0/1 labels and how the logistic function turns a linear score into a probability.
2. **Interpret** the logistic-regression model: the log-odds is linear in the features, so the decision boundary is a line (hyperplane), and $e^{w_i}$ is an odds ratio; **extend** it to R classes with R − 1 weight vectors.
3. **Derive**, step by step, the conditional log-likelihood $l(W)$ and its gradient $\partial l/\partial w_i = \sum_\ell x_i^\ell\,(y^\ell - \hat P(Y^\ell{=}1\mid x^\ell))$, and **verify** both numerically.
4. **Implement** gradient ascent from scratch, **diagnose** step sizes that are too small or too large, and **explain** why concavity guarantees the global optimum ("Mount Fuji, not the Himalayas").
5. **Derive** the MAP (L2-regularised) update from a Gaussian prior and **demonstrate** that on separable data the unregularised weights grow without bound while the regularised ones converge.
6. **Apply** logistic regression to real leukaemia data with genes selected on the training patients only, and **evaluate** it with a confusion matrix, precision and recall.
7. **Compare** generative (naïve Bayes) and discriminative (logistic regression) classifiers: what each models, how they relate mathematically, how they behave as the training set grows, and how well calibrated they are.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · Why a new classifier? | naïve Bayes vs logistic regression; why not linear regression (slides 2–5) | 12 min |
| 2 · The logistic function | a smooth, probabilistic sign function (slides 9–13) | 12 min |
| 3 · The probability model | log-odds, linear decision boundary, odds ratios (slides 7–17, 19–20) | 15 min |
| 4 · More than two classes | R − 1 weight vectors, softmax, iris (slide 18) | 10 min |
| 5 · Training by MCLE | derive l(W) and its gradient; concavity; gradient ascent (slides 22–30) | 30 min |
| 6 · MAP and separable data | Gaussian prior = L2 penalty; weights that run away (slides 31–32) | 15 min |
| 7 · Real data: ALL vs AML | 10 training-selected genes, confusion matrix, precision, recall (slide 36) | 20 min |
| 8 · Generative vs discriminative | NB ⇒ logistic form; learning curves; calibration (slides 33–35) | 20 min |
| 9 · Limits of a linear boundary | quadratic features; pointers to trees, SVMs, neural nets (slide 32) | 8 min |
| Wrap-up | takeaways, checkpoint, discussion, lab note | 3 min |

## 🧫 The biological question

A haematologist looking at a new leukaemia sample needs more than a label. "AML, with probability 0.97" and "AML,
with probability 0.55" call for different actions — the second one asks for more tests before a treatment is chosen.
What we want from a classifier is a well-founded **probability** $P(Y = \text{AML} \mid \text{expression profile})$.

In [Module 08](08_knn_naive_bayes.ipynb) naïve Bayes produced such probabilities indirectly: it modelled how each
leukaemia type *generates* expression values, $P(X \mid Y)$, and inverted that model with Bayes' rule. We saw the
price: when genes are correlated — and genes in the same pathway always are — naïve Bayes counts the same evidence
several times and becomes over-confident. **Logistic regression** takes the direct route: it models
$P(Y \mid X)$ itself, with one weight per gene, and learns the weights so that the predicted probabilities match the
observed outcomes as well as possible. It is the workhorse of biomedical prediction — clinical risk scores,
case–control association studies (the "odds ratios" reported by every GWAS of a binary disease), and the last layer
of most neural networks (Module 22).

We will derive it completely, train it with the algorithm that also trains deep networks — gradient ascent — and use
it on the same real leukaemia patients as in Module 08 (Golub et al. 1999), keeping the original training/test
split and choosing genes with the training patients only.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We need the logistic function `sigmoid` (written so that it never overflows), a stable log-sum-exp for the multi-class model, z-scoring, the Golub leukaemia loader, the evaluation metrics of slide 36, and a function that shades the decision regions of a 2-feature classifier. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_DATA_URL = "https://raw.githubusercontent.com/xulabs/edu/main/ml/tutorial/BiologyToML/datasets/{name}"


CACHE_DIR = os.environ.get("BIO2ML_CACHE", "course_data")


def fetch_course_file(name: str, timeout: float = 60.0) -> str | None:
    """Return a local path to the course data file ``name``, or ``None`` if it cannot be found.

    Looks in the repository's ``datasets/`` folder first (when the notebook runs inside a
    clone of xulabs/edu), then in the download cache, and finally downloads it from GitHub.
    """
    for base in ("../datasets", "datasets", "ml/tutorial/BiologyToML/datasets", CACHE_DIR):
        path = os.path.join(base, name)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
    dest = os.path.join(CACHE_DIR, name)
    os.makedirs(CACHE_DIR, exist_ok=True)
    try:
        with urllib.request.urlopen(REPO_DATA_URL.format(name=name), timeout=timeout) as r, \
                open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return dest
    except Exception as e:  # noqa: BLE001 — any network problem → offline fallback
        print(f"⚠️  Could not obtain {name} ({type(e).__name__}); using a simulated stand-in instead.")
        return None


def simulate_golub_like(rng: np.random.Generator) -> dict:
    """Simulated stand-in for the Golub data: 72 samples (47 'ALL', 25 'AML') × 3,571 genes.

    Log10-scale expression with gene-specific means, correlated gene modules and
    60 genes that differ between the classes.  Same keys as ``load_golub``.
    """
    n_all, n_aml, J = 47, 25, 3571
    y = np.r_[np.zeros(n_all, int), np.ones(n_aml, int)]
    base = rng.normal(2.6, 0.45, J)
    modules = rng.normal(0, 1, (len(y), 20))                  # 20 hidden "programmes"
    loading = rng.normal(0, 0.15, (20, J)) * (rng.random((20, J)) < 0.1)
    X = base + modules @ loading + rng.normal(0, 0.25, (len(y), J))
    informative = rng.choice(J, 60, replace=False)
    X[:, informative] += np.outer(y - 0.5, rng.choice([-1, 1], 60) * rng.uniform(0.4, 0.9, 60))
    is_train = np.r_[np.arange(n_all) < 27, np.arange(n_aml) < 11]
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL"), genes=np.array([f"gene{j:04d}" for j in range(J)]),
                is_train=is_train, sample_id=np.arange(1, len(y) + 1), source="SIMULATED stand-in (offline)")


def load_golub() -> dict:
    """Golub et al. (1999) leukaemia expression data, preprocessed as in Dudoit et al. (2002).

    Returns a dict with ``X`` (72 × 3,571, log10 expression), ``y`` (0 = ALL, 1 = AML),
    ``labels``, ``genes`` (Affymetrix probe IDs), ``is_train`` (the original 38/34 split),
    ``sample_id`` and ``source``.  Falls back to ``simulate_golub_like`` when offline.

    Raw intensities were clipped to [100, 16000] before the log10, so the value 2.0 means
    "at or below the detection limit" (about 20 % of all entries).  Some genes are therefore
    constant within a class — per-class variance estimates need a small floor.
    """
    path = fetch_course_file("golub_leukemia.csv.gz")
    if path is None:
        return simulate_golub_like(np.random.default_rng(1999))
    df = pd.read_csv(path)
    return dict(X=df.iloc[:, 3:].to_numpy(float), y=(df["label"] == "AML").to_numpy(int),
                labels=df["label"].to_numpy(), genes=np.array(df.columns[3:]),
                is_train=(df["golub_split"] == "train").to_numpy(), sample_id=df["sample_id"].to_numpy(),
                source="Golub et al. 1999 (real data)")


def sigmoid(z):
    """Logistic function 1 / (1 + e^(−z)), written via tanh so that it never overflows."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, float)))


def logsumexp(a, axis=None):
    """log(Σ exp(a)) computed stably (subtract the maximum first)."""
    a = np.asarray(a, float)
    m = np.max(a, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True)) + m
    return np.squeeze(out, axis=axis) if axis is not None else out.item()


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


def precision_recall_f1(y_true, y_pred):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 (0 when undefined)."""
    c = confusion_counts(y_true, y_pred)
    precision = c["TP"] / (c["TP"] + c["FP"]) if c["TP"] + c["FP"] else 0.0
    recall = c["TP"] / (c["TP"] + c["FN"]) if c["TP"] + c["FN"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_decision_regions(predict, X, y, ax=None, resolution: int = 200, title: str = "",
                          class_names=None, pad: float = 0.5):
    """Shade the regions a 2-feature classifier assigns to each class and overlay the data.

    ``predict`` maps an (M, 2) array to M integer labels 0, 1, 2, …
    """
    X, y = np.asarray(X, float), np.asarray(y).astype(int)
    ax = ax or plt.gca()
    x0, x1 = X[:, 0].min() - pad, X[:, 0].max() + pad
    y0, y1 = X[:, 1].min() - pad, X[:, 1].max() + pad
    gx, gy = np.meshgrid(np.linspace(x0, x1, resolution), np.linspace(y0, y1, resolution))
    Z = np.asarray(predict(np.c_[gx.ravel(), gy.ravel()])).astype(int).reshape(gx.shape)
    k = max(int(y.max()), int(Z.max())) + 1
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(CLASS_COLORS[:k])
    ax.contourf(gx, gy, Z, levels=np.arange(k + 1) - 0.5, cmap=cmap, alpha=0.18)
    ax.contour(gx, gy, Z, levels=np.arange(k) + 0.5, colors="k", linewidths=0.8)
    for c in range(k):
        m = y == c
        if m.any():
            ax.scatter(X[m, 0], X[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", linewidth=0.3,
                       label=(class_names[c] if class_names is not None else f"class {c}"))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['fetch_course_file', 'simulate_golub_like', 'load_golub', 'sigmoid', 'logsumexp', 'standardize', 'accuracy', 'confusion_counts', 'precision_recall_f1', 'plot_decision_regions']))

---
## 1 · Why a new classifier?

> 📍 **Where we are.** Module 08 introduced the three classifier types of 02-620 (slide 2): k-NN (non-parametric —
> "no parameters mean flexibility"), naïve Bayes (parametric, generative: models $P(Y, X \mid \theta) =
> P(Y)P(X\mid Y,\theta)$) and logistic regression (parametric, discriminative: models $P(Y \mid X, \theta)$). This
> notebook is about the third. The slide asks: *why logistic regression?*

Slides 3–4 contrast the two pictures on the same grid of "+" ($Y = 0$) and "o" ($Y = 1$) samples. Naïve Bayes
describes *where each class lives* — a distribution of $X$ for the "+" and one for the "o" — and the boundary
between the classes is a by-product. Logistic regression goes straight for the **boundary**: it draws a line and
says how the probability of "o" changes as you cross it. If all we need is $P(Y \mid X)$, modelling
$P(X \mid Y)$ — often the harder problem, with many correlated features — may be an unnecessary detour.

**Why not just use linear regression?** (slide 5). We could code the classes as $y \in \{0, 1\}$ and fit a straight
line $y \approx w_0 + w_1 x$ by least squares (Module 06), then call "1" whenever the line exceeds 0.5. Let us try it
on a SIMULATED biomarker: 20 healthy people and 20 patients, plus — in the second panel — six patients with
extremely high marker levels (very advanced disease). Those six are *easy* cases: nobody doubts their diagnosis.

In [ ]:
# ── 1.1 Linear regression vs logistic regression on 0/1 labels (SIMULATED biomarker) ──
rng = np.random.default_rng(SEED + 5)
x_h, x_d = rng.normal(1.0, 0.45, 20), rng.normal(2.2, 0.45, 20)       # log marker level: healthy, diseased
x_bm, y_bm = np.r_[x_h, x_d], np.r_[np.zeros(20), np.ones(20)]
x_more, y_more = np.r_[x_bm, rng.normal(9.0, 0.6, 6)], np.r_[y_bm, np.ones(6)]   # + 6 very high, clearly diseased

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (x, y, title) in zip(axes, [(x_bm, y_bm, "40 patients"), (x_more, y_more, "the same 40 + 6 extreme patients")]):
    b1, b0 = np.polyfit(x, y, 1)                                       # least-squares line (Module 06)
    thr_lin = (0.5 - b0) / b1                                          # where the line crosses 0.5
    lr = LogisticRegression(C=1e8).fit(x[:, None], y)                  # preview: we derive this in Section 5
    thr_log = -lr.intercept_[0] / lr.coef_[0, 0]                       # where P(Y=1|x) = 0.5
    grid = np.linspace(x.min() - 0.5, x.max() + 0.5, 300)
    ax.scatter(x, y + rng.uniform(-0.03, 0.03, len(y)), s=16, c=np.where(y == 1, "tab:red", "tab:blue"), zorder=3)
    ax.plot(grid, b0 + b1 * grid, color="tab:green", lw=2, label=f"linear regression (cut {thr_lin:.2f})")
    ax.plot(grid, lr.predict_proba(grid[:, None])[:, 1], color="k", lw=2, label=f"logistic regression (cut {thr_log:.2f})")
    ax.axvline(thr_lin, color="tab:green", ls="--", lw=1); ax.axvline(thr_log, color="k", ls=":", lw=1)
    ax.axhline(0.5, color="gray", lw=0.5)
    n_lin, n_log = np.sum((x > thr_lin) != y), np.sum((x > thr_log) != y)
    ax.set_title(f"{title}: errors linear {n_lin}, logistic {n_log}")
    ax.set_xlabel("log biomarker level"); ax.set_ylabel("y (0 = healthy, 1 = diseased)"); ax.legend(fontsize=7)
    print(f"{title:33s}: linear cut {thr_lin:.2f} ({n_lin} errors), logistic cut {thr_log:.2f} ({n_log} errors); "
          f"linear predictions range {(b0 + b1 * x).min():+.2f} … {(b0 + b1 * x).max():+.2f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** On the first 40 patients both methods put the cut near 1.5 and make the same 3 errors.
Adding six *obviously* diseased patients should change nothing — yet the least-squares line tilts to reach them, its
0.5-crossing moves right (from 1.46 to 1.76) and it now misclassifies 7 patients. The logistic curve barely moves
(cut 1.55 in both panels): once a point is far on the correct side, its predicted probability is already ≈ 1 and it
stops pulling. Two further problems of the straight line are visible: it predicts "probabilities" below 0 and above 1
(from −0.28 in the first panel up to 1.28 in the second), and its squared-error criterion assumes Gaussian noise around the line, which makes no sense for
an outcome that can only be 0 or 1.

What we need is a function that maps a linear score $w_0 + w_1 x$ (any real number) to a probability between 0 and 1.

### 📑 Slide connection — `004_classification-LR`, slides 2–5

- **What the slides show.** The three classifier types again (k-NN: no parameters, flexibility; naïve Bayes: $P(Y, X\mid\theta) = P(Y)P(X\mid Y,\theta)$; logistic regression: $P(Y\mid X,\theta)$ — "why logistic regression?"); the same 2-D data with the naïve-Bayes view (slide 3) and the logistic-regression view with a single straight boundary (slide 4); a straight line vs an S-shaped logistic fit to 0/1 outcomes (slide 5).
- **What this notebook adds.** A runnable version of slide 5 that shows *why* the straight line fails: easy, extreme cases drag the least-squares line and move the 0.5 cut (7 errors instead of 3), predictions leave [0, 1], and the logistic fit is unaffected.
- **Exam tip.** If asked "why not linear regression for classification?", give three reasons: predictions outside [0, 1], sensitivity to points far on the correct side, and a Gaussian-noise model that does not fit 0/1 outcomes.

---
## 2 · The logistic function: a smooth, probabilistic sign function (slides 9–13)

> 📍 **Where we are.** We want to squash any real number into the interval (0, 1). The logistic function does it.

🧮 **Math refresher — exponentials and logarithms** (Module 04). $e \approx 2.718$; $e^b$ ("exp of b") is positive for
every $b$, equals 1 at $b = 0$, grows very fast for positive $b$ and shrinks towards 0 for negative $b$. The natural
logarithm $\ln$ undoes it: $\ln e^b = b$. Rules we will use: $e^{a+b} = e^a e^b$, $e^{-b} = 1/e^b$,
$\ln(ab) = \ln a + \ln b$, $\ln(a/b) = \ln a - \ln b$, $\ln 1 = 0$. (In this course `log` and `ln` both mean the natural
logarithm.)

The **logistic function** (also called the sigmoid) is (slide 10)

$$a \;=\; \sigma(b) \;=\; \frac{1}{1 + \exp(-b)} .$$

*In words:* feed in any number $b$; if $b$ is very negative, $e^{-b}$ is huge and $a$ is close to 0; if $b$ is very
positive, $e^{-b}$ is tiny and $a$ is close to 1; at $b = 0$, $e^0 = 1$ and $a = 1/2$. Fill in slide 10's blanks:
"when $b = 0$, $a = $ …", "when $b \to \infty$ …", "when $b \to -\infty$ …".

In [ ]:
# ── 2.1 Values of the logistic function (the blanks of slide 10) ──
for b in [-50, -5, -2, -1, 0, 1, 2, 5, 50]:
    print(f"b = {b:+4d}:  σ(b) = {sigmoid(b):.6f}")
print("\nσ(−b) = 1 − σ(b)?", np.allclose(sigmoid(-np.linspace(-5, 5, 11)), 1 - sigmoid(np.linspace(-5, 5, 11))))

🔍 **What just happened?** $\sigma(0) = 0.5$, $\sigma(b) \to 1$ as $b \to \infty$ and $\to 0$ as $b \to -\infty$
(at $b = \pm 50$ it is 1 or 0 to six decimals). The function is symmetric: $\sigma(-b) = 1 - \sigma(b)$, which is why
$P(Y = 0 \mid X)$ will turn out to be the logistic of *minus* the score.

**From a hard to a soft classifier** (slides 11–13). The simplest linear classifier computes the score
$x = w_0 + \sum_i w_i X_i$ and outputs $y = \operatorname{sign}(x)$: $+1$ if $x > 0$, $-1$ if $x < 0$. That is a step
function: it says nothing about *how sure* we are, and it has no useful derivative (flat everywhere, a jump at 0), so
we cannot improve the weights by following a slope. Slide 13 replaces the step by the logistic function: a smooth
step that turns the score into a probability.

In [ ]:
# ── 2.2 The logistic function as a smooth version of the step (sign) function ──
b = np.linspace(-6, 6, 601)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.3))
a1.plot(b, (b > 0).astype(float), color="gray", lw=3, alpha=0.5, label="step: 1 if score > 0 else 0")
for c, col in zip([1, 3, 10], ["tab:blue", "tab:orange", "tab:red"]):
    a1.plot(b, sigmoid(c * b), color=col, lw=1.8, label=f"σ({c}·b)")
a1.set_xlabel("score b = w0 + Σ wi xi"); a1.set_ylabel("P(Y = 1 | x)"); a1.legend(fontsize=7)
a1.set_title("larger weights → steeper, more confident")
a2.plot(b, np.sign(b), color="gray", lw=3, alpha=0.5, label="sign(b)  (slide 11)")
a2.plot(b, 2 * sigmoid(b) - 1, color="tab:blue", lw=1.8, label="2σ(b) − 1")
a2.set_xlabel("score b"); a2.set_ylabel("output in [−1, 1]"); a2.legend(fontsize=7)
a2.set_title("the logistic is a rescaled, smoothed sign function")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Multiplying the score by a larger number makes the logistic curve steeper; in the limit it
becomes the step function. So the *size* of the weights encodes confidence — remember this for Section 6, where on
some data the weights want to become infinitely large. The right panel shows that $2\sigma(b) - 1$ is a smoothed
version of slide 11's sign function.

**The derivative** — needed for gradient ascent. Write $\sigma(b) = (1 + e^{-b})^{-1}$ and use the chain rule
(Module 04): the outer function $u^{-1}$ has derivative $-u^{-2}$, the inner $u = 1 + e^{-b}$ has derivative
$-e^{-b}$. So

$$\sigma'(b) \;=\; \frac{e^{-b}}{(1 + e^{-b})^2} \;=\; \underbrace{\frac{1}{1 + e^{-b}}}_{\sigma(b)}\cdot\underbrace{\frac{e^{-b}}{1 + e^{-b}}}_{1 - \sigma(b)} \;=\; \sigma(b)\,\big(1 - \sigma(b)\big).$$

*In words:* the slope of the logistic curve is largest (1/4) where $\sigma = 1/2$, at the decision boundary, and
nearly zero far from it, where the prediction is already confident.

In [ ]:
# ── 2.3 Check σ'(b) = σ(b)(1 − σ(b)) with finite differences ──
b_test = np.array([-4.0, -1.0, 0.0, 0.5, 3.0])
h = 1e-6
numeric = (sigmoid(b_test + h) - sigmoid(b_test - h)) / (2 * h)      # slope of a tiny secant
formula = sigmoid(b_test) * (1 - sigmoid(b_test))
print(pd.DataFrame({"b": b_test, "finite difference": numeric, "σ(1 − σ)": formula}).round(6).to_string(index=False))
print("largest difference:", f"{np.abs(numeric - formula).max():.1e}")

🔍 **What just happened?** The numerical slope and the formula agree to about $10^{-11}$; the slope peaks at 0.25 for
$b = 0$.

⚠️ **A numerical pitfall.** Written naively, `1 / (1 + np.exp(-b))` overflows for $b \approx -710$ ($e^{710}$ exceeds
the largest floating-point number) and NumPy prints warnings. The toolkit's `sigmoid` uses the identity
$\sigma(b) = \tfrac12\,(1 + \tanh(b/2))$, which never overflows. Likewise we will compute $\ln(1 + e^{z})$ with
`np.logaddexp(0, z)` rather than `np.log(1 + np.exp(z))`.

---
## 3 · The probability model and its linear decision boundary (slides 7–17, 19–20)

> 📍 **Where we are.** We combine a linear score — as in linear regression — with the logistic function.

**Set-up** (slide 7): input features $X = \langle X_1, \dots, X_n \rangle$ (the slides write $n$ for the number of
features; elsewhere in the course we call it $J$), binary output $Y$. The model is

$$P(Y = 1 \mid X) \;=\; \frac{\exp\big(w_0 + \sum_{i=1}^n w_i X_i\big)}{1 + \exp\big(w_0 + \sum_{i=1}^n w_i X_i\big)} \;=\; \sigma\Big(w_0 + \sum_{i=1}^n w_i X_i\Big),
\qquad P(Y = 0 \mid X) \;=\; \frac{1}{1 + \exp\big(w_0 + \sum_{i} w_i X_i\big)} .$$

*In words:* compute a weighted sum of the features — the "regression" part — and pass it through the logistic
function — the "logistic" part (slide 9). The two probabilities add up to 1 (slide 8). The two forms of
$P(Y = 1\mid X)$ are equal: divide numerator and denominator by $e^{z}$ to get $1/(1 + e^{-z})$.

⚠️ **A sign convention you will meet on the slides.** Slides 8–19 (and 32) write $P(Y{=}1 \mid X) = 1/(1 + \exp(w_0 + \sum_i
w_iX_i))$ — the convention of Mitchell's textbook — whereas slides 23–31 (the likelihood and gradient) use
$P(Y{=}1 \mid X) = \exp(w_0 + \sum_i w_iX_i)/(1 + \exp(\cdot))$, as above. The two describe exactly the same family of
models: one is the other with **every weight's sign flipped** ($w \to -w$). We use the second convention throughout
(it is also scikit-learn's), so that a positive weight means "this feature raises $P(Y = 1)$". Under the first
convention slide 17 reads $\ln \frac{P(Y=0\mid X)}{P(Y=1\mid X)} = w_0 + \sum_i w_iX_i$; under ours the same
statement becomes $\ln \frac{P(Y=1\mid X)}{P(Y=0\mid X)} = w_0 + \sum_i w_iX_i$, derived next.

🧮 **Odds and log-odds.** The **odds** of an event with probability $p$ are $p/(1-p)$ (probability 0.8 = odds 4 to 1),
familiar from epidemiology. Divide the two probabilities of the model:

$$\frac{P(Y=1\mid X)}{P(Y=0\mid X)} \;=\; \exp\Big(w_0 + \sum_i w_i X_i\Big) \quad\Longrightarrow\quad \ln\frac{P(Y=1\mid X)}{P(Y=0\mid X)} \;=\; w_0 + \sum_{i=1}^n w_i X_i .$$

*In words:* the **log-odds is a linear function of the features** — "linear model!" (slide 17). Logistic regression
is linear regression for the log-odds. Two consequences:

1. **Odds ratios.** Increasing $X_i$ by one unit adds $w_i$ to the log-odds, i.e. multiplies the odds by $e^{w_i}$. In
   a case–control GWAS with genotype coded 0/1/2, $e^{w}$ is the per-allele **odds ratio** reported in the papers.
2. **A linear decision boundary** (slide 14). We predict $Y = 1$ when $P(Y = 1\mid X) > 0.5$, i.e. when the odds exceed
   1, i.e. when the log-odds is positive:

$$\hat Y = 1 \iff w_0 + \sum_{i=1}^n w_i X_i > 0 .$$

*In words:* the boundary $w_0 + \sum_i w_iX_i = 0$ is a straight line in 2-D, a plane in 3-D and a "hyperplane" in
more dimensions. The weight vector $(w_1, \dots, w_n)$ points perpendicular to it, towards the $Y = 1$ side.

Slide 15 shows $\text{logistic}(w_1x_1 + w_2x_2)$ for ten weight vectors. We reproduce it, as heat maps.

In [ ]:
# ── 3.1 Slide 15: logistic(w1 x1 + w2 x2) for various W = (w1, w2) ──
W_SLIDE15 = [(1, 4), (5, 4), (-2, 3), (0, 2), (2, 2), (5, 1), (1, 0), (3, 0), (-2, -1), (2, -2)]
g = np.linspace(-10, 10, 201)
G1, G2 = np.meshgrid(g, g)
fig, axes = plt.subplots(2, 5, figsize=(12, 5.0))
for ax, (w1, w2) in zip(axes.ravel(), W_SLIDE15):
    im = ax.imshow(sigmoid(w1 * G1 + w2 * G2), origin="lower", extent=(-10, 10, -10, 10), cmap="RdBu_r", vmin=0, vmax=1)
    ax.contour(G1, G2, w1 * G1 + w2 * G2, levels=[0], colors="k", linestyles="--", linewidths=1)  # boundary
    ax.annotate("", xy=(4 * w1 / np.hypot(w1, w2), 4 * w2 / np.hypot(w1, w2)), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color="k", lw=1.5))                         # direction of w
    ax.set_title(f"W = ({w1}, {w2}),  |W| = {np.hypot(w1, w2):.1f}", fontsize=8)
    ax.set_xticks([-10, 0, 10]); ax.set_yticks([-10, 0, 10]); ax.tick_params(labelsize=7)
fig.colorbar(im, ax=axes, shrink=0.8, label="P(Y = 1 | x1, x2)")
fig.suptitle("logistic(w1·x1 + w2·x2): the arrow is W, the dashed line the boundary (P = 0.5)", fontsize=10)
plt.show()

🔍 **What just happened?** In every panel the dashed boundary passes through the origin (there is no $w_0$ here — an
intercept would shift the line away from the origin) and is perpendicular to the arrow $W$; the probability rises
from blue (0) to red (1) in the direction of $W$. The **direction** of $W$ sets the orientation of the boundary; its
**length** sets how sharp the transition is: compare $W = (1, 0)$ with $W = (3, 0)$, or $(2, 2)$ with $(5, 4)$.

**Classifying a new sample** (slide 20) is then a matter of evaluating $P(Y = 1 \mid X^{new})$ and
$P(Y = 0 \mid X^{new})$ and choosing the larger — equivalently, checking the sign of the score.

In [ ]:
# ── 3.2 Classify a new sample and check the log-odds identity ──
w0, w = -1.0, np.array([2.0, -1.0])            # an illustrative model with two features
x_new = np.array([1.5, 0.5])
z = w0 + w @ x_new                              # the score: w0 + Σ wi xi
p1 = np.exp(z) / (1 + np.exp(z))                # P(Y=1 | x), slide-23 form
p0 = 1 / (1 + np.exp(z))                        # P(Y=0 | x)
print(f"score z = {w0} + 2.0·1.5 − 1.0·0.5 = {z:.2f}")
print(f"P(Y=1|x) = {p1:.4f} (σ(z) = {sigmoid(z):.4f}),  P(Y=0|x) = {p0:.4f},  sum = {p1 + p0:.4f}")
print(f"log-odds ln(P1/P0) = {np.log(p1 / p0):.4f}  (= z)  →  predict Y = {int(p1 > p0)}")
print(f"odds ratio for +1 unit of feature 1: e^w1 = {np.exp(w[0]):.2f};  feature 2: e^w2 = {np.exp(w[1]):.2f}")

🔍 **What just happened?** The score is 1.5, so $P(Y = 1 \mid x) = \sigma(1.5) = 0.818$ and we predict class 1; the
log-odds equals the score exactly, as derived. One more unit of feature 1 would multiply the odds of class 1 by
$e^{2} \approx 7.4$; one more unit of feature 2 would multiply them by $e^{-1} \approx 0.37$.

⚠️ **Odds ratios are conditional and not causal.** $e^{w_i}$ describes the change in odds when $X_i$ changes *and all
other features in the model stay fixed*; adding or removing correlated features changes it. A large weight on a gene
means the gene's expression helps predict the class, not that the gene causes the disease.

✏️ **Your turn.** Change `x_new` to a point exactly on the boundary (for example $x = (0.5, 0)$, or any point with
$2x_1 - x_2 = 1$). What are $P(Y = 1 \mid x)$ and the log-odds there?

---
## 4 · More than two classes (slide 18)

> 📍 **Where we are.** So far $Y \in \{0, 1\}$ ($R = 2$ classes). Leukaemia has subtypes (B-ALL, T-ALL, AML…), cells
> have many types. How does the model generalise to $R > 2$ classes $\{y_1, \dots, y_R\}$?

Slide 18: learn $R - 1$ sets of weights, one per class except the last, and set

$$P(Y = y_k \mid X) = \frac{\exp\big(w_{k0} + \sum_{i} w_{ki}X_i\big)}{1 + \sum_{j=1}^{R-1}\exp\big(w_{j0} + \sum_{i} w_{ji}X_i\big)}\;\; (k < R), \qquad P(Y = y_R \mid X) = \frac{1}{1 + \sum_{j=1}^{R-1}\exp\big(w_{j0} + \sum_{i} w_{ji}X_i\big)} .$$

*In words:* each class except the last gets its own score; exponentiate the scores (making them positive), give the
reference class $y_R$ the score 0 (so $e^0 = 1$, the "1 +" in the denominator), and divide by the total so that the $R$
probabilities add to 1. For $R = 2$ this is exactly the binary model.

🧮 **Why only R − 1 weight vectors?** Adding the same vector to *all* class scores multiplies every numerator and the
denominator by the same factor, leaving the probabilities unchanged. So one class's weights can be fixed at zero
without loss. Many software packages (scikit-learn included) instead keep $R$ weight vectors and normalise with the
**softmax** $P(y_k \mid x) = e^{s_k}/\sum_{j=1}^{R} e^{s_j}$; subtracting the last class's weights from all of them
gives the slide's form. We check this on a real 3-class data set: Fisher's iris flowers (three species, measured
petal length and width).

In [ ]:
# ── 4.1 Three classes: iris species from two petal measurements (real data, scikit-learn) ──
iris = load_iris()
X_iris, y_iris = iris.data[:, 2:4], iris.target                  # petal length, petal width (cm); 3 species
sk_multi = LogisticRegression(C=1e4, max_iter=5000).fit(X_iris, y_iris)   # softmax with R = 3 weight vectors
W_all = np.c_[sk_multi.intercept_, sk_multi.coef_]                # shape (R, 1 + n)
W_slide = W_all[:-1] - W_all[-1]                                  # R − 1 vectors relative to the last class


def multiclass_proba(X, W):
    '''Slide 18: R − 1 weight vectors (rows of W, intercept first); the last class has score 0.'''
    X1 = np.c_[np.ones(len(X)), X]
    scores = np.c_[X1 @ W.T, np.zeros(len(X))]                    # (N, R): last column = reference class
    return np.exp(scores - logsumexp(scores, axis=1)[:, None])    # normalise stably


P_slide = multiclass_proba(X_iris, W_slide)
print("largest difference to scikit-learn's softmax probabilities:", f"{np.abs(P_slide - sk_multi.predict_proba(X_iris)).max():.1e}")
k_unsure = int(np.argmin(P_slide.max(axis=1)))                  # the flower the model is least sure about
print(f"least certain flower (#{k_unsure}, truly {iris.target_names[y_iris[k_unsure]]}): " +
      ", ".join(f"P({n}) = {p:.3f}" for n, p in zip(iris.target_names, P_slide[k_unsure])) +
      f";  sum = {P_slide[k_unsure].sum():.6f}")
print(f"training accuracy: {accuracy(y_iris, P_slide.argmax(axis=1)):.3f}")
fig, ax = plt.subplots(figsize=(5, 3.8))
plot_decision_regions(lambda Q: multiclass_proba(Q, W_slide).argmax(axis=1), X_iris, y_iris, ax=ax, pad=0.3,
                      class_names=list(iris.target_names))
ax.set_xlabel("petal length (cm)"); ax.set_ylabel("petal width (cm)"); ax.legend(fontsize=7)
ax.set_title("3-class logistic regression: piecewise-linear boundaries"); plt.tight_layout(); plt.show()

🔍 **What just happened?** The slide's $R - 1$ parameterisation reproduces scikit-learn's $R$-vector softmax
exactly (to floating-point precision): the redundant third vector carried no information. The three probabilities of
each flower add up to 1 — even for the flower the model is least sure about, which it splits between versicolor and
virginica — and the boundaries between species are straight line segments — each pairwise boundary is
where two class scores are equal, a linear equation. Setosa is separated perfectly; versicolor and virginica overlap a
little.

(Fitting the multi-class model uses the same gradient-ascent idea as Section 5; each class's gradient has the same
form, features × (indicator of the class − predicted probability). We used scikit-learn here only to obtain weights.)

### 📑 Slide connection — `004_classification-LR`, slides 7–20

- **What the slides show.** The model $P(Y{=}1\mid X) = 1/(1+\exp(w_0+\sum_i w_iX_i))$ and $P(Y{=}0\mid X)$ as its complement (slides 7–9, Mitchell's sign convention); the logistic function $a = 1/(1+\exp(-b))$ with its limits (slides 10, 12); sign(x) as the hard classifier it smooths (slides 11, 13); the linear boundary $w_0+\sum_i w_ix_i = 0$ (slide 14); logistic$(w_1x_1+w_2x_2)$ surfaces for ten $W$ (slide 15); the log odds is linear — "linear model!" (slides 16–17); R-class logistic regression with R − 1 weight vectors (slide 18); classifying $X^{new}$ (slide 20).
- **What this notebook adds.** Values and derivative of the logistic function checked numerically; the sign-convention switch between slides 8–19 and 23–31 made explicit (flip all weights); odds ratios and their caveats; slide 15 as heat maps with the weight vector drawn; the equivalence of the slide's R − 1 form and the softmax, verified on real iris data.
- **Exam tip.** Be able to derive "log-odds is linear" in two lines and to state the decision rule as the sign of $w_0 + \sum_i w_iX_i$. Watch which class the slide's formula calls $Y = 1$ — the sign of every weight depends on it.

---
## 5 · Training: maximum conditional likelihood (slides 22–30)

> 📍 **Where we are.** Step 2 of the course's two-step template: estimate $W = (w_0, w_1, \dots, w_n)$ from $L$
> training examples $\{\langle X^1, Y^1\rangle, \dots, \langle X^L, Y^L\rangle\}$. Naïve Bayes maximised the *joint*
> likelihood $\prod_\ell P(Y^\ell, X^\ell)$ (Module 08). Logistic regression only models $P(Y \mid X)$, so it maximises
> the **conditional** likelihood (MCLE, slide 22):

$$W_{MCLE} \;=\; \arg\max_W \prod_{\ell=1}^{L} P(Y^\ell \mid X^\ell, W), \qquad l(W) \;\equiv\; \ln\prod_{\ell} P(Y^\ell\mid X^\ell, W) \;=\; \sum_{\ell=1}^{L}\ln P(Y^\ell \mid X^\ell, W).$$

*In words:* choose the weights under which the observed outcomes are most probable *given* the observed features.
We never ask how probable the features themselves were.

To derive and visualise everything we use a small SIMULATED data set with a known answer: for $L = 200$ cells we
"measure" one standardised signalling-protein level $x$, and each cell responds ($y = 1$) with probability
$\sigma(-0.5 + 1.5x)$. The true weights are therefore $w_0 = -0.5$, $w_1 = 1.5$.

In [ ]:
# ── 5.1 A SIMULATED data set drawn from a logistic model with known weights ──
rng = np.random.default_rng(SEED + 54)
L_cells = 200
W_TRUE = np.array([-0.5, 1.5])
x_sig = rng.normal(0, 1, L_cells)                                   # standardised protein level
y_sig = (rng.random(L_cells) < sigmoid(W_TRUE[0] + W_TRUE[1] * x_sig)).astype(float)
X1_sig = np.c_[np.ones(L_cells), x_sig]                             # column of ones carries w0 (Module 06)
print(f"{L_cells} cells, {int(y_sig.sum())} responders;  X1 has shape {X1_sig.shape} (intercept column + 1 feature)")

### 5.1 Expressing the conditional log-likelihood (slides 23–25)

**Step 1 — one formula for both outcomes.** Because $Y^\ell$ is 0 or 1, we can write the log-probability of either
outcome with one expression (slide 23):

$$\ln P(Y^\ell \mid X^\ell, W) \;=\; Y^\ell \ln P(Y^\ell{=}1 \mid X^\ell, W) + (1 - Y^\ell)\ln P(Y^\ell{=}0 \mid X^\ell, W).$$

*In words:* for a sample with $Y^\ell = 1$ the second term vanishes and we keep $\ln P(Y = 1)$; for $Y^\ell = 0$ the
first vanishes — the same on/off trick as the Bernoulli formula $\pi^Y(1 - \pi)^{1-Y}$ of Module 08, after taking logs.

**Step 2 — collect the $Y^\ell$ terms** (slide 24): expand $(1 - Y^\ell)\ln P_0 = \ln P_0 - Y^\ell \ln P_0$ and combine
with $Y^\ell\ln P_1$ using $\ln a - \ln b = \ln(a/b)$:

$$l(W) \;=\; \sum_{\ell}\Big[\,Y^\ell \ln\frac{P(Y^\ell{=}1\mid X^\ell, W)}{P(Y^\ell{=}0\mid X^\ell, W)} \;+\; \ln P(Y^\ell{=}0 \mid X^\ell, W)\Big].$$

*In words:* each sample contributes its log-odds (if it is a "1") plus the log-probability of "0".

**Step 3 — insert the model** (slide 25). The log-odds is the score $z^\ell = w_0 + \sum_i w_i X_i^\ell$ (Section 3),
and $\ln P(Y = 0 \mid X, W) = \ln 1 - \ln\big(1 + \exp(z)\big) = -\ln(1 + e^{z})$. Therefore

$$\boxed{\,l(W) \;=\; \sum_{\ell=1}^{L}\Big[\,Y^\ell\Big(w_0 + \sum_{i=1}^n w_i X_i^\ell\Big) \;-\; \ln\Big(1 + \exp\big(w_0 + \sum_{i=1}^n w_i X_i^\ell\big)\Big)\Big]\,}$$

*In words:* reward the score of every responder, and subtract for every sample a term that grows with its score —
the model gains by giving high scores to the "1"s and low scores to the "0"s. Let us check numerically that the
direct formula of step 1 and the final formula of step 3 agree.

In [ ]:
# ── 5.2 The conditional log-likelihood, two ways ──
def cll_direct(w, X1, y):
    '''Step 1: Σ [y ln P(Y=1|x) + (1 − y) ln P(Y=0|x)].'''
    p = sigmoid(X1 @ w)
    return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))


def cond_log_likelihood(w, X1, y):
    '''Step 3 (slide 25): Σ [y z − ln(1 + e^z)] with z = X1 w; ln(1 + e^z) computed stably with logaddexp.'''
    z = X1 @ w
    return np.sum(y * z - np.logaddexp(0, z))


for w_try in [np.array([0.0, 0.0]), np.array([-0.5, 1.5]), np.array([2.0, -1.0])]:
    print(f"W = {w_try}:  step-1 formula {cll_direct(w_try, X1_sig, y_sig):10.4f}   "
          f"step-3 formula {cond_log_likelihood(w_try, X1_sig, y_sig):10.4f}")
print(f"reference: L·ln(0.5) = {L_cells * np.log(0.5):.4f} (W = 0 predicts 0.5 for everybody)")

🔍 **What just happened?** The two formulas agree for every $W$, so the algebra of slides 23–25 is right. With
$W = 0$ every cell gets probability 0.5 and $l = 200\ln 0.5 = -138.6$; the true weights score much higher; weights
pointing the wrong way score much lower. The log-likelihood is always negative (a sum of logs of probabilities) and
would be 0 only if every outcome were predicted with certainty.

### 5.2 The gradient (slide 27)

There is **no closed-form solution**: setting the derivatives to zero gives equations that cannot be solved with
algebra (slide 26, "bad news"). We will climb the log-likelihood instead, and for that we need its slope. Take the
partial derivative of $l(W)$ with respect to one weight $w_i$, using $\partial z^\ell / \partial w_i = X_i^\ell$
(with $X_0^\ell = 1$ for the intercept) and the two rules on slide 27, $(\ln f)' = f'/f$ and $(e^f)' = e^f f'$:

$$\frac{\partial}{\partial w_i}\Big[Y^\ell z^\ell\Big] = Y^\ell X_i^\ell, \qquad
\frac{\partial}{\partial w_i}\ln\big(1 + e^{z^\ell}\big) = \frac{e^{z^\ell}\,X_i^\ell}{1 + e^{z^\ell}} = \hat P(Y^\ell{=}1\mid X^\ell, W)\,X_i^\ell .$$

*In words:* the first term's slope is the feature value (for responders); the second term's slope is the feature
value times the model's current predicted probability. Subtracting:

$$\boxed{\,\frac{\partial l(W)}{\partial w_i} \;=\; \sum_{\ell=1}^{L} X_i^\ell\,\Big(Y^\ell - \hat P(Y^\ell{=}1 \mid X^\ell, W)\Big)\,}$$

*In words:* for each sample, compare what happened ($Y^\ell$, 0 or 1) with what the model predicted
($\hat P$) — the **prediction error** — and weight that error by the sample's value of feature $i$. If responders
with high $x_i$ are under-predicted, the gradient says: increase $w_i$. In matrix form, with $X$ the $L \times (n+1)$
matrix including the column of ones, all derivatives at once are $\nabla l = X^\top(y - \hat p)$. (Compare linear
regression in Module 06: $X^\top(y - Xw)$ — the same "features × errors" shape.)

In [ ]:
# ── 5.3 The gradient, checked against finite differences ──
def cll_gradient(w, X1, y):
    '''Slide 27: ∂l/∂w_i = Σ_ℓ x_i^ℓ (y^ℓ − P̂(Y=1 | x^ℓ)), for all i at once: X1ᵀ (y − p̂).'''
    return X1.T @ (y - sigmoid(X1 @ w))


w_try, h = np.array([0.3, 0.8]), 1e-6
numeric = np.array([(cond_log_likelihood(w_try + h * e, X1_sig, y_sig) - cond_log_likelihood(w_try - h * e, X1_sig, y_sig))
                    / (2 * h) for e in np.eye(2)])
print("analytic gradient :", cll_gradient(w_try, X1_sig, y_sig).round(5))
print("finite differences:", numeric.round(5))

🔍 **What just happened?** The derived gradient matches the numerical slopes to five decimals: at $W = (0.3, 0.8)$
the log-likelihood would increase by lowering $w_0$ and raising $w_1$ — towards the true values $(-0.5, 1.5)$.

### 5.3 Concavity: Mount Fuji instead of the Himalayas (slides 26, 30)

"Good news: $l(W)$ is a **concave** function of $W$" (slide 26). A function is concave if the straight line (chord)
between any two points of its graph lies *below* the graph:

$$f\big(t x_1 + (1-t)x_2\big) \;\ge\; t f(x_1) + (1-t) f(x_2)\quad\text{for all } t\in[0,1].$$

*In words:* the surface bends downwards everywhere — a single dome, with no dips and no second peak. (Convex is the
mirror image: the chord lies above, a single valley.) For logistic regression this follows from the second
derivative: differentiating the gradient once more gives
$\partial^2 l/\partial w_i^2 = -\sum_\ell (X_i^\ell)^2\,\hat P^\ell(1 - \hat P^\ell)$, a sum of negative terms, and the
same calculation along *any* direction $v$ in weight space gives $-\sum_\ell (v^\top X^\ell)^2\,\hat P^\ell(1 - \hat
P^\ell) \le 0$ — so every slice of the surface curves downwards. Consequence: **any peak we climb to is the highest peak** — gradient
ascent cannot get stuck on a lower summit ("Mount Fuji instead of Himalayas", slide 30). Let us look at the surface.

In [ ]:
# ── 5.4 The log-likelihood surface over (w0, w1): one smooth dome ──
g0, g1 = np.linspace(-3, 2, 121), np.linspace(-1, 4, 121)
W0g, W1g = np.meshgrid(g0, g1)
Z = W0g[..., None] + W1g[..., None] * x_sig                         # scores for every grid point and cell
surface = np.sum(y_sig * Z - np.logaddexp(0, Z), axis=2)             # l(w0, w1) on the grid
i_best = np.unravel_index(np.argmax(surface), surface.shape)
print(f"grid maximum near w0 = {W0g[i_best]:.3f}, w1 = {W1g[i_best]:.3f};  l = {surface.max():.2f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
cs = a1.contour(W0g, W1g, surface, levels=np.linspace(surface.max() - 120, surface.max(), 13), cmap="viridis")
a1.clabel(cs, fontsize=6, fmt="%.0f")
a1.plot(*W_TRUE, "r*", ms=12, label="true weights"); a1.plot(W0g[i_best], W1g[i_best], "kx", ms=9, label="grid maximum")
a1.set_xlabel("w0 (intercept)"); a1.set_ylabel("w1 (protein weight)"); a1.legend(fontsize=7)
a1.set_title("contours of l(w0, w1): a single peak")
w1_line = np.linspace(-1, 4, 200)
slice_l = [cond_log_likelihood(np.array([W0g[i_best], v]), X1_sig, y_sig) for v in w1_line]
a2.plot(w1_line, slice_l, color="tab:blue", lw=2, label="l along w1 (w0 at its best value)")
pa, pb = 0.0, 3.5
la, lb = (cond_log_likelihood(np.array([W0g[i_best], v]), X1_sig, y_sig) for v in (pa, pb))
a2.plot([pa, pb], [la, lb], "k--", lw=1, label="a chord lies below the curve")
a2.set_xlabel("w1"); a2.set_ylabel("conditional log-likelihood"); a2.legend(fontsize=7)
a2.set_title("concave: the curve bends downwards everywhere")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The contours are nested closed curves around a single maximum, near — but not exactly at
— the true weights (red star): with 200 cells the estimate carries sampling error (Module 05). Along any line through
the surface the log-likelihood is a downward-bending curve, and the chord between two of its points lies below it.

### 5.4 Gradient ascent (slides 28–29)

To climb a concave hill, repeatedly take a step uphill (slide 28), until the improvement
$|l(W^{(t-1)}) - l(W^{(t)})|$ falls below a small tolerance $\varepsilon$:

$$w_i \;\leftarrow\; w_i + \eta\,\frac{\partial l(W)}{\partial w_i} \;=\; w_i + \eta \sum_{\ell} X_i^\ell\big(Y^\ell - \hat P(Y^\ell{=}1\mid X^\ell, W)\big)\quad\text{for all } i .$$

*In words:* move every weight a little in the direction that increases the log-likelihood, by an amount
proportional to its slope; stop when the improvements become negligible. The **step size** $\eta$ (slide 29 calls it
$\lambda$) matters: too small and we need very many iterations, too large and we overshoot the peak.

🧮 Note that the gradient is a *sum* over samples, so its size grows with $L$; a good $\eta$ for 200 samples is too
large for 20,000. (Many implementations divide by $L$ and use the mean instead.)

In [ ]:
# ── 5.5 Gradient ascent from scratch, with four step sizes ──
def gradient_ascent(X1, y, eta, n_iter=300, lam=0.0, tol=1e-9, w_init=None):
    '''Slide 28: w ← w + η ∂/∂w [l(W) − λ Σ_{i≥1} w_i²]; stop when |Δ objective| < tol.

    lam = 0 is maximum conditional likelihood; lam > 0 is the MAP version of Section 6 (w0 not penalised).
    Returns the final weights, the objective after every step and the path of the weights.
    '''
    w = np.zeros(X1.shape[1]) if w_init is None else np.array(w_init, float)
    objective = lambda v: cond_log_likelihood(v, X1, y) - lam * np.sum(v[1:] ** 2)
    hist, path = [objective(w)], [w.copy()]
    for t in range(n_iter):
        grad = cll_gradient(w, X1, y)
        grad[1:] -= 2 * lam * w[1:]                    # gradient of the log-prior (Section 6)
        w = w + eta * grad
        hist.append(objective(w)); path.append(w.copy())
        if abs(hist[-1] - hist[-2]) < tol:
            break
    return w, np.array(hist), np.array(path)


start = [1.5, -0.5]                                    # a deliberately poor starting point
runs = {eta: gradient_ascent(X1_sig, y_sig, eta, w_init=start) for eta in [0.001, 0.005, 0.03, 0.065]}
for eta, (w_hat, hist, path) in runs.items():
    print(f"η = {eta:<6}: stopped after {len(hist) - 1:3d} steps, w = ({w_hat[0]:+.3f}, {w_hat[1]:+.3f}), l = {hist[-1]:.3f}")

In [ ]:
# ── 5.6 Learning curves and paths on the likelihood surface ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
cols = {0.001: "tab:blue", 0.005: "tab:green", 0.03: "tab:orange", 0.065: "tab:red"}
for eta, (w_hat, hist, path) in runs.items():
    a1.plot(np.arange(1, len(hist) + 1), hist, color=cols[eta], lw=1.5, label=f"η = {eta}")
    a2.plot(path[:40, 0], path[:40, 1], "o-", color=cols[eta], ms=2.5, lw=1, label=f"η = {eta} (first 40 steps)")
a1.axhline(surface.max(), color="gray", ls=":", lw=1)
a1.set_xscale("log"); a1.set_ylim(surface.max() - 60, surface.max() + 5)
a1.set_xlabel("iteration + 1 (log scale)"); a1.set_ylabel("l(W)"); a1.legend(fontsize=7)
a1.set_title("too small = slow, too large = oscillation")
a2.contour(W0g, W1g, surface, levels=np.linspace(surface.max() - 120, surface.max(), 13), cmap="Greys", linewidths=0.7)
a2.plot(*start, "ks", ms=6); a2.set_xlim(-3, 2); a2.set_ylim(-1, 4)
a2.set_xlabel("w0"); a2.set_ylabel("w1"); a2.legend(fontsize=7, loc="lower left"); a2.set_title("paths of gradient ascent")
plt.tight_layout(); plt.show()

🔍 **What just happened?** All runs start from the same poor point (black square). With $\eta = 0.001$ the climb is
steady but slow: it needs well over 100 steps to approach the top and still has not met the stopping rule after
300. $\eta = 0.005$ and $\eta = 0.03$ reach the peak
— the larger step in a handful of iterations. With $\eta = 0.065$ the steps overshoot: the path zig-zags across the
ridge and the log-likelihood settles at a *lower* value than the peak, because the iterates jump back and forth
across the summit forever instead of converging (slide 29: "λ↑: overshoot, λ↓: too many iterations"). Concavity
guarantees a single peak, not that a badly tuned algorithm reaches it.

Finally, scikit-learn should find the same peak. It minimises $\frac{1}{2C}\lVert w\rVert^2 - l(W)$, so a huge $C$
means "no penalty".

In [ ]:
# ── 5.7 Check against scikit-learn ──
w_ours = runs[0.03][0]
sk_lr = LogisticRegression(C=1e8, tol=1e-10, max_iter=1000).fit(x_sig[:, None], y_sig)
print(f"ours        : w0 = {w_ours[0]:+.4f}, w1 = {w_ours[1]:+.4f}")
print(f"scikit-learn: w0 = {sk_lr.intercept_[0]:+.4f}, w1 = {sk_lr.coef_[0, 0]:+.4f}")
print(f"true values : w0 = {W_TRUE[0]:+.4f}, w1 = {W_TRUE[1]:+.4f}")

🔍 **What just happened?** Our 20-line gradient ascent and scikit-learn's optimiser (a quasi-Newton method, L-BFGS)
agree to four decimals — they climb the same concave hill. Both differ from the true weights by sampling error, which
would shrink if we simulated more cells.

✏️ **Your turn.** (1) Change `L_cells` to 2,000 in cell 5.1 and re-run 5.1–5.7: does $\eta = 0.03$ still converge?
Why not? (Hint: the gradient is a sum over samples.) (2) Start gradient ascent from three very different points;
do they end at the same weights?

---
## 6 · MAP estimation, regularisation and separable data (slides 31–32)

> 📍 **Where we are.** As in Modules 05 and 07, maximum likelihood can over-fit when data are scarce. The remedy is
> again a prior on the weights — MAP estimation (slide 31).

**A Gaussian prior on the weights**, $W \sim \mathcal N(0, \sigma^2 I)$: each $w_i$ ($i \ge 1$) is a priori near 0, with
spread $\sigma$. The MAP estimate maximises $\ln\big[P(W)\prod_\ell P(Y^\ell\mid X^\ell, W)\big] = \ln P(W) + l(W)$, and
$\ln P(W) = \sum_i \ln\big[\tfrac{1}{\sqrt{2\pi\sigma^2}}e^{-w_i^2/(2\sigma^2)}\big] = -\sum_i \frac{w_i^2}{2\sigma^2} + \text{const}$. So

$$W_{MAP} \;=\; \arg\max_W \Big[\, l(W) \;-\; \lambda\sum_{i=1}^{n} w_i^2 \Big], \qquad \lambda = \frac{1}{2\sigma^2},
\qquad w_i \;\leftarrow\; w_i - \eta\,2\lambda w_i + \eta\sum_\ell X_i^\ell\big(Y^\ell - \hat P(Y^\ell{=}1\mid X^\ell, W)\big).$$

*In words:* the prior adds a **penalty** that grows with the squared size of the weights (the L2 or "ridge"
penalty of Module 07), and every gradient step first shrinks each weight a little towards zero ("weight decay").
A small prior variance $\sigma^2$ means a large $\lambda$ and strong shrinkage. As in ridge regression, the intercept
$w_0$ is usually not penalised.

⚠️ **A factor of 2 on slide 31.** The slide defines $\lambda = 1/(2\sigma^2)$ but writes the update as
$w_i \leftarrow w_i - \eta\lambda w_i + \dots$. The derivative of $\lambda w_i^2$ is $2\lambda w_i$, so with
$\lambda = 1/(2\sigma^2)$ the shrinkage term is $\eta\,2\lambda w_i = \eta w_i/\sigma^2$ (equivalently, the slide's
update is right if the penalty is written $\tfrac{\lambda}{2}\sum_i w_i^2$ with $\lambda = 1/\sigma^2$). The factor
only rescales $\lambda$; the idea is unchanged. Our code uses $l(W) - \lambda\sum_i w_i^2$ with gradient
$-2\lambda w_i$, which corresponds to scikit-learn's `C` $= 1/(2\lambda)$.

**When does regularisation become essential?** Slide 32 asks: *what if the positive and negative samples are (or
are not) completely separable?* If a line separates the training classes perfectly, we can always increase the
likelihood by making the weights larger — the logistic curve becomes steeper (Section 2), every training probability
moves closer to 0 or 1, and $l(W)$ creeps towards its upper limit 0 without ever reaching it. **The MLE does not
exist** (it is "at infinity"). This happens easily with few samples and many features — exactly the situation of
expression data. Let us see it on two real Golub genes.

In [ ]:
# ── 6.1 Two real Golub genes, chosen on the TRAINING patients: the training set is separable ──
golub = load_golub()
print("Data source:", golub["source"])
Xg, yg, is_tr = golub["X"], golub["y"].astype(float), golub["is_train"]
Xg_tr, yg_tr, Xg_te, yg_te = Xg[is_tr], yg[is_tr], Xg[~is_tr], yg[~is_tr]


def signal_to_noise(X, y):
    '''Golub et al.'s per-gene score (μ_AML − μ_ALL) / (σ_AML + σ_ALL), as in Module 08.'''
    a, b = X[y == 1], X[y == 0]
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.nan_to_num((a.mean(axis=0) - b.mean(axis=0)) / (a.std(axis=0) + b.std(axis=0)))


rank = np.argsort(-np.abs(signal_to_noise(Xg_tr, yg_tr)))          # 38 training patients only
g2 = rank[:2]
Z2_tr, m2, s2 = standardize(Xg_tr[:, g2]); Z2_te, _, _ = standardize(Xg_te[:, g2], m2, s2)
Z2_tr1, Z2_te1 = np.c_[np.ones(len(Z2_tr)), Z2_tr], np.c_[np.ones(len(Z2_te)), Z2_te]
print("genes:", list(golub["genes"][g2]), f"| {len(yg_tr)} training patients ({int(yg_tr.sum())} AML)")

norms, fits = {}, {}
for lam in [0.0, 0.01, 0.1, 1.0]:
    w_hat, hist, path = gradient_ascent(Z2_tr1, yg_tr, eta=0.05, n_iter=20_000, lam=lam, tol=0.0)
    norms[lam], fits[lam] = np.linalg.norm(path[:, 1:], axis=1), w_hat
    p_tr = sigmoid(Z2_tr1 @ w_hat)
    print(f"λ = {lam:<5}: |w| after 100 / 1,000 / 20,000 steps = {norms[lam][100]:5.2f} / {norms[lam][1000]:5.2f} / "
          f"{norms[lam][-1]:5.2f};  l(W) = {cond_log_likelihood(w_hat, Z2_tr1, yg_tr):8.4f};  "
          f"training P range {p_tr.min():.4f} – {p_tr.max():.4f}")
fig, ax = plt.subplots(figsize=(6, 3.4))
for lam, col in zip(norms, ["tab:red", "tab:orange", "tab:green", "tab:blue"]):
    ax.plot(np.arange(1, len(norms[lam])), norms[lam][1:], color=col, lw=2, label=f"λ = {lam}")
ax.set_xscale("log"); ax.set_xlabel("gradient-ascent iteration (log scale)"); ax.set_ylabel("|w| (gene weights)")
ax.set_title("separable training data: the MLE runs away, MAP converges"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With the two top genes (cystatin C and zyxin, Module 08) the 38 training patients are
**linearly separable**. Without a prior ($\lambda = 0$) the weight vector keeps growing — by roughly 4–6 units every
time the number of iterations is multiplied by 10, the signature of slow (logarithmic) growth towards infinity — while
$l(W)$ approaches 0 and the training probabilities are pushed to 0.000… and 0.999…: the model claims certainty about
every training patient. With any $\lambda > 0$ the weights level off at a finite value; larger $\lambda$, smaller
weights and more moderate probabilities.

What does this do to predictions for *new* patients?

In [ ]:
# ── 6.2 Probability contours: the unregularised model is a cliff, the MAP model a slope ──
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
g = np.linspace(-2.6, 3.2, 200)
Ga, Gb = np.meshgrid(g, g)
for ax, lam in zip(axes, [0.0, 1.0]):
    P = sigmoid(fits[lam][0] + fits[lam][1] * Ga + fits[lam][2] * Gb)
    im = ax.contourf(Ga, Gb, P, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.6)
    cs = ax.contour(Ga, Gb, P, levels=[0.1, 0.5, 0.9], colors="k", linewidths=[0.7, 1.5, 0.7], linestyles=["--", "-", "--"])
    ax.clabel(cs, fontsize=7, fmt="%.1f")
    for c, col in [(0, "tab:blue"), (1, "tab:red")]:
        ax.scatter(*Z2_tr[yg_tr == c].T, s=18, color=col, edgecolor="k", lw=0.3)
        ax.scatter(*Z2_te[yg_te == c].T, s=34, facecolor="none", edgecolor=col, marker="D", lw=1.2)
    p_te = sigmoid(Z2_te1 @ fits[lam])
    ll_te = -np.mean(yg_te * np.log(np.clip(p_te, 1e-15, 1)) + (1 - yg_te) * np.log(np.clip(1 - p_te, 1e-15, 1)))
    ax.set_title(f"λ = {lam}: |w| = {np.linalg.norm(fits[lam][1:]):.1f}, test errors {int(np.sum((p_te > 0.5) != yg_te))}/34, "
                 f"test log-loss {ll_te:.2f}", fontsize=9)
    ax.set_xlabel(f"{golub['genes'][g2[0]]} (standardised)"); ax.set_ylabel(f"{golub['genes'][g2[1]]} (standardised)")
fig.colorbar(im, ax=axes, shrink=0.85, label="P(AML | x)")
fig.suptitle("dots: training patients; diamonds: test patients (blue ALL, red AML)", fontsize=9)
plt.show()

🔍 **What just happened?** Without regularisation, the 0.1 and 0.9 contours are squeezed against the 0.5 line: the
model jumps from "certainly ALL" to "certainly AML" over a tiny distance. Test patients falling near the boundary
receive extreme, over-confident probabilities, and a confident mistake costs a lot of **log-loss** (the average of
$-\ln$ of the probability given to the true class — Module 08, Section 9; lower is better, 0.69 = always saying 0.5). With $\lambda = 1$ the
transition is gradual, the boundary moves a little, and the probabilities near the boundary are honest about the
uncertainty. Compare the two titles: the regularised model happens to make one *more* error (3 instead of 2 of 34 —
a single patient near the boundary), but its test log-loss is three times lower (0.18 vs 0.54). Regularisation
matters most for the *probabilities*. (The test patients were not used to choose $\lambda$: we simply compare two
fixed values.)

### 📑 Slide connection — `004_classification-LR`, slides 22–32

- **What the slides show.** MCLE: $W_{MCLE} = \arg\max_W\prod_\ell P(Y^\ell\mid W, X^\ell)$ and $l(W) = \sum_\ell \ln P(Y^\ell\mid X^\ell, W)$ (slide 22); splitting the $Y^\ell=1$ and $Y^\ell=0$ terms, rewriting with the log-odds, and the final $l(W) = \sum_\ell Y^\ell(w_0+\sum_i w_iX_i^\ell) - \ln(1+\exp(w_0+\sum_i w_iX_i^\ell))$ (slides 23–25); no closed form but concave (slide 26); the gradient $\sum_\ell X_i^\ell(Y^\ell - \hat P(Y^\ell=1\mid X^\ell, W))$ (slide 27); gradient ascent with step size η and stopping rule (slides 28–29); "Mount Fuji instead of Himalayas" (slide 30); MAP with $W\sim N(0,\sigma^2 I)$ adds $-\eta\lambda w_i$, $\lambda = 1/(2\sigma^2)$ (slide 31); separable vs non-separable data (slide 32).
- **What this notebook adds.** Every algebra step written out and checked numerically (two forms of $l(W)$, the gradient vs finite differences); the second derivative that proves concavity; the likelihood surface; four step sizes including a non-converging one; agreement with scikit-learn; the factor-2 inconsistency of slide 31 resolved; and the separability problem demonstrated on real leukaemia data.
- **Exam tip.** The derivation of $l(W)$ and of $\partial l/\partial w_i$ is the most-asked exam derivation of this lecture: practise it on paper until you can do it without notes. Know why separable data make the MLE infinite and how the Gaussian prior fixes it.

---
## 7 · Real data: diagnosing ALL vs AML from ten genes (slide 36)

> 📍 **Where we are.** We have a trained, regularised classifier. Now the full workflow on real patients, with the
> evaluation measures of slide 36.

**Protocol** (every choice made *before* looking at the test patients):

1. Rank the 3,571 genes by the signal-to-noise score on the **38 training patients** and keep the top **10** (a
   "handful" fixed in advance).
2. Standardise each gene with the training mean and standard deviation.
3. Fit logistic regression by gradient ascent with the prior $\sigma^2 = 1$ — a belief that a one-standard-deviation
   change in one gene rarely shifts the log-odds by more than about ±2 — i.e. $\lambda = 1/(2\sigma^2) = 0.5$.
4. Predict the **34 test patients** once and evaluate.

In [ ]:
# ── 7.1 Select ten genes on the training patients and fit the MAP logistic regression ──
G = 10
g10 = rank[:G]
KNOWN = {"M27891_at": "cystatin C (CST3)", "X95735_at": "zyxin (ZYX)", "M23197_at": "CD33",
         "U50136_rna1_at": "leukotriene C4 synthase (LTC4S)",
         "U22376_cds2_s_at": "MYB", "M84526_at": "complement factor D (CFD)"}
Z_tr, m10, s10 = standardize(Xg_tr[:, g10]); Z_te, _, _ = standardize(Xg_te[:, g10], m10, s10)
Z_tr1, Z_te1 = np.c_[np.ones(len(Z_tr)), Z_tr], np.c_[np.ones(len(Z_te)), Z_te]
LAM = 0.5                                                           # σ² = 1  →  λ = 1/(2σ²)
w_lr, hist_lr, _ = gradient_ascent(Z_tr1, yg_tr, eta=0.01, n_iter=20_000, lam=LAM, tol=1e-12)
sk10 = LogisticRegression(C=1 / (2 * LAM), tol=1e-10, max_iter=10_000).fit(Z_tr, yg_tr)
print(f"gradient ascent converged after {len(hist_lr) - 1} steps; largest difference to scikit-learn (C = 1/(2λ) = 1): "
      f"{np.abs(w_lr - np.r_[sk10.intercept_, sk10.coef_[0]]).max():.1e}")
print(pd.DataFrame({"gene": golub["genes"][g10], "annotation": [KNOWN.get(n, "") for n in golub["genes"][g10]],
                    "weight (per SD)": w_lr[1:].round(3), "odds ratio per SD": np.exp(w_lr[1:]).round(2)}).to_string(index=False))
print(f"intercept w0 = {w_lr[0]:.3f}")

🔍 **What just happened?** Our gradient ascent reproduces scikit-learn's solution (to about $10^{-5}$), confirming
the correspondence $C = 1/(2\lambda)$. Nine of the ten genes are higher in AML and get positive weights; MYB, higher in
ALL, gets a negative one. Each weight is a change in log-odds per standard deviation of the gene: for example an odds
ratio of about 1.8 means one SD more expression multiplies the odds of AML by 1.8, *with the other nine genes held
fixed*. The weights are modest (all below 1): the ten genes are strongly correlated with each other, and the
penalty spreads the evidence among them instead of letting a few genes carry huge weights.

**Measuring accuracy** (slide 36). With AML as the "positive" class, every test patient falls in one cell of the
**confusion matrix**: true positives (TP, AML called AML), false positives (FP, ALL called AML), false negatives (FN,
AML called ALL) and true negatives (TN). Then

$$\text{precision} = \frac{TP}{TP + FP}, \qquad \text{recall} = \frac{TP}{TP + FN}.$$

*In words:* **precision** — of the patients we called "AML", how many really have AML? **Recall** (sensitivity) — of
the patients who really have AML, how many did we catch? A classifier that calls everybody AML has perfect recall
and poor precision; one that calls only the single most obvious case AML has high precision and poor recall.

In [ ]:
# ── 7.2 Evaluate on the 34 independent test patients ──
p_te = sigmoid(Z_te1 @ w_lr)
pred_te = (p_te > 0.5).astype(int)
cc = confusion_counts(yg_te, pred_te)
prec, rec, f1 = precision_recall_f1(yg_te, pred_te)
print("confusion matrix (rows = truth, columns = prediction):")
print(pd.DataFrame([[cc["TN"], cc["FP"]], [cc["FN"], cc["TP"]]], index=["true ALL", "true AML"],
                   columns=["predicted ALL", "predicted AML"]).to_string())
print(f"\naccuracy {accuracy(yg_te, pred_te):.3f} | precision {prec:.3f} | recall {rec:.3f} | F1 {f1:.3f}")
for i in np.flatnonzero(pred_te != yg_te):
    print(f"misclassified: test patient {golub['sample_id'][~is_tr][i]} (truly {'AML' if yg_te[i] else 'ALL'}), "
          f"P(AML | x) = {p_te[i]:.3f}")

fig, ax = plt.subplots(figsize=(6.5, 2.6))
for c, col, name in [(0, "tab:blue", "ALL"), (1, "tab:red", "AML")]:
    m = yg_te == c
    ax.scatter(p_te[m], c + rng.uniform(-0.15, 0.15, m.sum()), color=col, s=30, edgecolor="k", lw=0.3, label=f"true {name}")
ax.axvline(0.5, color="k", ls="--", lw=1)
ax.set_yticks([0, 1]); ax.set_yticklabels(["true ALL", "true AML"]); ax.set_xlim(-0.02, 1.02)
ax.set_xlabel("predicted P(AML | 10 genes)"); ax.set_title(f"34 independent test patients — {golub['source']}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** 33 of the 34 independent test patients are classified correctly (accuracy 0.971, against
a baseline of 0.588). All 20 ALL patients are called ALL, so every "AML" call is right: precision 1.0. One AML patient
is called ALL (recall 13/14 = 0.929). The probabilities are informative rather than all-or-nothing: most patients
sit near 0 or 1, but the misclassified AML patient gets $P(\text{AML}) \approx 0.1$ — a wrong call, but not an
absurdly confident one (the same patient was also missed by Gaussian naïve Bayes in Module 08).

⚠️ **How much should you trust 0.971?** With 34 test patients one error is the difference between 0.971 and 1.0;
the uncertainty of an accuracy estimated from 34 cases is several percentage points. The original study also selected
genes and a model on the training patients only — had we selected the genes with all 72 patients, the test result
would be optimistically biased (Ambroise & McLachlan 2002; Module 23).

✏️ **Your turn.** Change `G` to 2, 5 or 50, or `LAM` to 0.05 or 5, and re-run 7.1–7.2. How do the weights, the errors
and the probabilities change? Remember that choosing `G` or `LAM` *because* they give the best test result would be
using the test set for training — in a real study you would choose them by cross-validation on the 38 training
patients (Module 23).

---
## 8 · Generative vs discriminative classifiers (slides 33–35)

> 📍 **Where we are.** We now have one classifier of each kind. Slide 33 asks: *what are the differences, pros and
> cons?*

| | **Generative** (e.g. naïve Bayes) | **Discriminative** (e.g. logistic regression) |
|---|---|---|
| models | $P(X \mid Y)$ and $P(Y)$, i.e. the joint $P(X, Y)$ | $P(Y \mid X)$ only |
| fitting criterion | $\theta = \arg\max\prod_\ell P(y^\ell, x^\ell \mid \theta)$ (joint likelihood) | $\theta = \arg\max\prod_\ell P(y^\ell \mid x^\ell, \theta)$ (conditional likelihood) |
| learning | closed form (counts, means, variances) | iterative (gradient ascent) |
| assumptions | a form for how $X$ is generated (e.g. Gaussian, conditionally independent features) | a form for $P(Y\mid X)$ (log-odds linear in $X$) |
| pros | fast; handles missing features; can generate data; can need less data | fewer assumptions; robust to correlated features; usually better probabilities and lower asymptotic error |
| cons | wrong assumptions about $X$ bias $P(Y \mid X)$ (e.g. double counting) | needs iterative optimisation; says nothing about $X$ itself |

**They are closer than they look.** Take Gaussian naïve Bayes with variances **shared** by both classes,
$P(X_i \mid Y = k) = \mathcal N(\mu_{ik}, \sigma_i^2)$. Its log-odds is

$$\ln\frac{P(Y{=}1\mid x)}{P(Y{=}0\mid x)} = \ln\frac{\pi}{1-\pi} + \sum_i \frac{(x_i - \mu_{i0})^2 - (x_i - \mu_{i1})^2}{2\sigma_i^2}
= \underbrace{\ln\frac{\pi}{1-\pi} + \sum_i \frac{\mu_{i0}^2 - \mu_{i1}^2}{2\sigma_i^2}}_{w_0} + \sum_i \underbrace{\frac{\mu_{i1} - \mu_{i0}}{\sigma_i^2}}_{w_i}\, x_i .$$

*In words:* expanding the squares, the $x_i^2$ terms cancel (because the variances are shared) and what remains is
**linear in $x$** — exactly the logistic-regression form. So this naïve Bayes model *is* a logistic regression, with
the weights computed from class means and variances instead of being fitted to maximise $P(Y \mid X)$. Same model
family, different estimation criterion. (With class-specific variances the $x_i^2$ terms survive and the boundary is
quadratic, as in Module 08.) Let us verify this on the ten Golub genes and compare the two sets of weights.

In [ ]:
# ── 8.1 Gaussian naive Bayes with shared variances = a logistic regression with different weights ──
pi_hat = yg_tr.mean()
mu1, mu0 = Z_tr[yg_tr == 1].mean(axis=0), Z_tr[yg_tr == 0].mean(axis=0)
var_pooled = (Z_tr - np.where(yg_tr[:, None] == 1, mu1, mu0)).var(axis=0)      # shared within-class variance
w_nb = np.r_[np.log(pi_hat / (1 - pi_hat)) + np.sum((mu0 ** 2 - mu1 ** 2) / (2 * var_pooled)), (mu1 - mu0) / var_pooled]


def log_gauss(x, mu, var):
    '''log N(x; μ, σ²), elementwise.'''
    return -0.5 * np.log(2 * np.pi * var) - (x - mu) ** 2 / (2 * var)


lj1 = np.log(pi_hat) + log_gauss(Z_te, mu1, var_pooled).sum(axis=1)       # naive Bayes computed the generative way
lj0 = np.log(1 - pi_hat) + log_gauss(Z_te, mu0, var_pooled).sum(axis=1)
print("largest difference: NB posterior via Bayes' rule vs σ(w0 + wᵀx) with the formula weights:",
      f"{np.abs(sigmoid(lj1 - lj0) - sigmoid(Z_te1 @ w_nb)).max():.1e}")
print(pd.DataFrame({"gene": golub["genes"][g10], "naive-Bayes weight": w_nb[1:].round(2),
                    "logistic-regression weight": w_lr[1:].round(2)}).to_string(index=False))
for name, w_model in [("naive Bayes (shared var.)", w_nb), ("logistic regression (MAP)", w_lr)]:
    log_odds = Z_te1 @ w_model
    pc = np.clip(sigmoid(log_odds), 1e-15, 1 - 1e-15)
    print(f"{name:26s}: test errors {int(np.sum((log_odds > 0) != yg_te))}/34, "
          f"test log-loss {-np.mean(yg_te * np.log(pc) + (1 - yg_te) * np.log(1 - pc)):.3f}, "
          f"median |log-odds| {np.median(np.abs(log_odds)):.1f}")

🔍 **What just happened?** The naïve-Bayes posterior computed the generative way (class densities and Bayes' rule)
equals $\sigma(w_0 + w^\top x)$ with the formula weights, to machine precision — the algebra holds. But the two weight
vectors are very different: naïve Bayes gives every gene a weight around 3–7, logistic regression around 0.1–0.75.
Naïve Bayes treats the ten correlated genes as ten *independent* witnesses and adds up all their evidence; logistic
regression, fitted to $P(Y\mid X)$, discovers that the genes largely repeat each other. Both classify the test patients
equally well (one error each), but naïve Bayes' log-odds are about ten times larger: its probabilities are extreme,
and its test log-loss is several times worse. Module 08's "subtlety #2" on real data.

A clean way to see the difference: **duplicate a feature**.

In [ ]:
# ── 8.2 Duplicated features: naive Bayes double-counts, logistic regression splits the weight ──
print("copies | NB total weight on x | LR weights (no penalty)        | LR P(Y=1 | x = 1) | NB P(Y=1 | x = 1)")
for k in [1, 2, 4]:
    Xk = np.c_[np.ones(L_cells), np.repeat(x_sig[:, None], k, axis=1)]          # the protein level k times
    w_k = gradient_ascent(Xk, y_sig, eta=0.03 / k, n_iter=5000)[0]              # MCLE, no penalty
    m1_, m0_ = x_sig[y_sig == 1].mean(), x_sig[y_sig == 0].mean()
    v_ = np.mean((x_sig - np.where(y_sig == 1, m1_, m0_)) ** 2)
    nb_slope = k * (m1_ - m0_) / v_                                             # each copy adds the same evidence
    nb_w0 = np.log(y_sig.mean() / (1 - y_sig.mean())) + k * (m0_ ** 2 - m1_ ** 2) / (2 * v_)
    print(f"  {k}    |        {nb_slope:5.2f}         | {str(np.round(w_k[1:], 3)):30s} |      {sigmoid(w_k[0] + w_k[1:].sum()):.3f}"
          f"        |      {sigmoid(nb_w0 + nb_slope):.3f}")

🔍 **What just happened?** With $k$ identical copies of the protein level, unpenalised logistic regression splits the
weight equally among the copies (they receive identical gradients) and its predictions do not change at all — the
conditional likelihood only cares about the total. Naïve Bayes adds the full evidence once per copy: its total weight
grows $k$-fold and its probability for a cell with $x = 1$ becomes more and more extreme, although no new information
was added. (With an L2 penalty, logistic regression would still split the weight, but the total would grow slightly,
because $k$ small weights are penalised less than one large one.)

### 8.1 Which one needs more data? (Ng & Jordan 2002)

Ng and Jordan compared naïve Bayes and logistic regression as the training set grows and argued that naïve Bayes
approaches its (higher) asymptotic error *faster*, whereas logistic regression has the *lower* asymptotic error — so
naïve Bayes can win with few samples and logistic regression with many. Let us test this honestly on the Wisconsin
diagnostic breast-cancer data (569 fine-needle aspirates, malignant vs benign; we use the ten "mean" nucleus
features), with 30 random training sets of each size and the remaining samples as test set. We compare Gaussian NB,
logistic regression with a negligible penalty (close to Ng & Jordan's unregularised version) and our MAP version
($\lambda = 0.5$, i.e. `C = 1`).

In [ ]:
# ── 8.3 Learning curves: naive Bayes vs logistic regression on breast-cancer data (real, scikit-learn) ──
bc = load_breast_cancer()
X_bc, y_bc = bc.data[:, :10], 1 - bc.target                       # ten 'mean' features; 1 = malignant
rng_bc = np.random.default_rng(SEED + 83)
sizes, R_REP = [6, 10, 14, 20, 30, 40, 60, 100, 200, 400], 30
err = {k: np.zeros((len(sizes), R_REP)) for k in ["naive Bayes", "LR, almost no penalty (C = 10⁴)", "LR, MAP λ = 0.5 (C = 1)"]}
pooled = {"y": [], "naive Bayes": [], "logistic regression": []}   # test predictions at m = 200, for calibration
for r in range(R_REP):
    perm = rng_bc.permutation(len(y_bc))
    test, pool = perm[450:], perm[:450]                            # 119 test samples, never used for training
    pos, neg = pool[y_bc[pool] == 1], pool[y_bc[pool] == 0]
    for si, m in enumerate(sizes):
        n_pos = max(2, int(round(m * 0.37)))                       # keep the malignant fraction ≈ 37 %
        tr = np.r_[pos[:n_pos], neg[:m - n_pos]]
        Ztr_bc, mu_bc, sd_bc = standardize(X_bc[tr]); Zte_bc, _, _ = standardize(X_bc[test], mu_bc, sd_bc)
        models = {"naive Bayes": GaussianNB(), "LR, almost no penalty (C = 10⁴)": LogisticRegression(C=1e4, max_iter=5000),
                  "LR, MAP λ = 0.5 (C = 1)": LogisticRegression(C=1.0, max_iter=5000)}
        for name, model in models.items():
            model.fit(Ztr_bc, y_bc[tr])
            err[name][si, r] = 1 - model.score(Zte_bc, y_bc[test])
            if m == 200 and name != "LR, almost no penalty (C = 10⁴)":
                pooled["naive Bayes" if name == "naive Bayes" else "logistic regression"].append(model.predict_proba(Zte_bc)[:, 1])
        if m == 200:
            pooled["y"].append(y_bc[test])
print("training size m | " + " | ".join(err))
for si, m in enumerate(sizes):
    print(f"      {m:4d}      | " + " | ".join(f"{err[k][si].mean():.3f} ± {err[k][si].std() / np.sqrt(R_REP):.3f}" for k in err))

In [ ]:
# ── 8.4 Plot the learning curves ──
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for (name, e), col, mk in zip(err.items(), ["tab:green", "tab:orange", "tab:blue"], ["o", "s", "^"]):
    mean, se = e.mean(axis=1), e.std(axis=1) / np.sqrt(R_REP)
    ax.plot(sizes, mean, marker=mk, color=col, lw=1.8, ms=4, label=name)
    ax.fill_between(sizes, mean - se, mean + se, color=col, alpha=0.2)
ax.set_xscale("log"); ax.set_xlabel("training-set size m (log scale)"); ax.set_ylabel("test error (mean of 30 splits)")
ax.set_ylim(0, 0.4); ax.legend(fontsize=7)
ax.set_title("Wisconsin breast cancer, 10 features: generative vs discriminative")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The picture is more nuanced than a slogan, so read it carefully:

- **Naïve Bayes plateaus early.** Its error drops quickly and is essentially flat (about 8–9 %) from about 30
  training samples on: it reaches its asymptote fast, as Ng & Jordan predicted — and more data do not help, because
  the model's assumptions (independent Gaussian features; the ten nucleus features are strongly correlated) are wrong.
- **Nearly unregularised logistic regression** is worse than naïve Bayes from about 10 to 60 training samples, equal
  at about 100, and better from 200 on (6.3 % vs 8.6 % error at 400) — exactly the crossover Ng & Jordan described.
  With only 6 samples naïve Bayes is the worst of all (22 % error): its variances are estimated from two to four
  samples per class.
- **Regularised logistic regression** (the MAP version of Section 6) is as good as or better than naïve Bayes at
  *every* size here: tied within the error bars at 10–14 samples, clearly better from about 40 on. The prior supplies
  exactly the stability that the unregularised discriminative model lacks with few samples.

So "generative models need less data" is a statement about *unregularised* models and depends on how wrong the
generative assumptions are; with a sensible prior, the discriminative model is usually the safer default — while
naïve Bayes remains attractive when it is fast enough and roughly right (and when features are missing at random,
which it handles naturally).

### 8.2 Calibration in brief

A classifier is **calibrated** if, among the patients to whom it assigns probability ≈ 0.8, about 80 % really are
positive. We pool the test predictions of the 30 splits at $m = 200$ and compare predicted probabilities with observed
frequencies in five bins (a reliability diagram).

In [ ]:
# ── 8.5 Reliability: predicted probability vs observed frequency (pooled test predictions, m = 200) ──
y_pool = np.concatenate(pooled["y"])
bins = np.linspace(0, 1, 6)
fig, ax = plt.subplots(figsize=(4.6, 3.8))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
for name, col in [("naive Bayes", "tab:green"), ("logistic regression", "tab:blue")]:
    p = np.concatenate(pooled[name])
    idx = np.clip(np.digitize(p, bins) - 1, 0, 4)
    centres = [p[idx == b].mean() for b in range(5) if np.any(idx == b)]
    freqs = [y_pool[idx == b].mean() for b in range(5) if np.any(idx == b)]
    pc = np.clip(p, 1e-15, 1 - 1e-15)
    ll = -np.mean(y_pool * np.log(pc) + (1 - y_pool) * np.log(1 - pc))
    ax.plot(centres, freqs, "o-", color=col, label=f"{name} (log-loss {ll:.2f})")
    print(f"{name:20s}: log-loss {ll:.3f};  fraction of predictions < 0.01 or > 0.99: {np.mean((p < 0.01) | (p > 0.99)):.2f}")
    print("   bins (mean predicted → observed): " + ", ".join(f"{c:.2f}→{f:.2f}" for c, f in zip(centres, freqs)))
ax.set_xlabel("mean predicted P(malignant) in bin"); ax.set_ylabel("observed fraction malignant")
ax.set_title("calibration on held-out samples (m = 200)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Logistic regression's points lie close to the diagonal: its probabilities mean what they
say, as expected from a model fitted to maximise the conditional likelihood (it is directly rewarded for honest
probabilities): its bins read 0.03→0.03, 0.29→0.29, 0.49→0.47, 0.70→0.75, 0.98→0.99. Naïve Bayes pushes 89 % of
its predictions below 0.01 or above 0.99, and they are not to be taken literally: of the samples it calls malignant
with probability ≈ 1.00, only 91 % are; of those it calls benign with ≈ 0.01, 6 % are malignant; and where it says
0.30, two thirds are malignant. Its log-loss is more than three times larger (0.53 vs 0.16). For risk communication — "your probability of malignancy is 30 %" —
this difference matters as much as accuracy. The repository's deep-dive notebook (🚀 Bonus) studies calibration with
Brier scores and reliability diagrams in detail.

### 📑 Slide connection — `004_classification-LR`, slides 33–37

- **What the slides show.** Generative vs discriminative: what are the differences, pros and cons (slide 33; speaker notes list fewer parameters and better task performance for discriminative models, data generation and insight into the data for generative ones); generative classifiers model $P(Y, X) = P(X\mid Y)P(Y)$ and use Bayes' rule, discriminative ones model $P(Y\mid X)$ directly (slides 34–35, with the joint vs conditional argmax); precision and recall with the TP/FP/FN/TN picture (slide 36); summary: NB closed-form MLE, LR iterative learning (slide 37).
- **What this notebook adds.** The derivation showing that Gaussian NB with shared variances has exactly the logistic form (with weights from means and variances), verified on real data; the weight comparison that exposes double counting; the duplicated-feature experiment; an honest Ng & Jordan learning-curve experiment on real breast-cancer data (the crossover appears only against nearly unregularised LR); a reliability diagram; and precision/recall on the Golub test set.
- **Exam tip.** Be able to fill in the generative/discriminative table, derive the shared-variance GNB log-odds, and compute precision and recall from a confusion matrix (and say which one a screening test should maximise).

---
## 9 · Limits of a linear boundary (slide 32)

> 📍 **Where we are.** Slide 32 adds a few "+" samples inside the "o" cloud (and an "o" among the "+"), asks what
> logistic regression does when the classes are *not* completely separable, and ends with "Limitations of logistic
> regression?". Its speaker notes answer "linear model won't work" for such data, and the slide hints at the fix: add
> squared terms $\sum_k w_k X_k^2$ to the score. Here we take the extreme case of that hint — one class surrounded by
> the other, so that no straight line can separate them.

Logistic regression is linear **in its features**, not necessarily in the original measurements. If we add squared
measurements as extra features, the boundary $w_0 + w_1x_1 + w_2x_2 + w_3x_1^2 + w_4x_2^2 = 0$ can be a circle or an
ellipse in the original coordinates. Biologically, this is the situation where a response happens only in an
*intermediate* range of two signals (a "Goldilocks" window), as for many dose–response relationships. We simulate
such data (SIMULATED: cells respond when the two signals are jointly moderate, with 5 % label noise).

In [ ]:
# ── 9.1 Ring-shaped classes: linear features vs added squared features (SIMULATED) ──
rng = np.random.default_rng(SEED + 91)
X_ring = rng.uniform(-3, 3, (400, 2))                               # two signal levels (standardised)
y_ring = (np.hypot(*X_ring.T) < 1.8).astype(float)                  # respond inside a disc
flip = rng.random(400) < 0.05
y_ring[flip] = 1 - y_ring[flip]                                     # 5 % label noise


def quad_features(X):
    '''Original features plus their squares: (x1, x2, x1², x2²).'''
    return np.c_[X, X ** 2]


fig, axes = plt.subplots(1, 2, figsize=(10, 4.0))
for ax, (name, feat) in zip(axes, [("linear features (x1, x2)", lambda X: X), ("+ squared features (x1², x2²)", quad_features)]):
    F, mu_f, sd_f = standardize(feat(X_ring))
    F1 = np.c_[np.ones(len(F)), F]
    w_f = gradient_ascent(F1, y_ring, eta=0.002, n_iter=5000, lam=0.01)[0]
    predict = lambda Q, w_f=w_f, feat=feat, mu_f=mu_f, sd_f=sd_f: (np.c_[np.ones(len(Q)), standardize(feat(Q), mu_f, sd_f)[0]] @ w_f > 0).astype(int)
    acc_f = accuracy(y_ring, predict(X_ring))
    plot_decision_regions(predict, X_ring, y_ring, ax=ax, pad=0.2, class_names=["no response", "response"])
    ax.set_title(f"{name}: training accuracy {acc_f:.2f}"); ax.set_xlabel("signal 1"); ax.set_ylabel("signal 2")
    print(f"{name:30s}: training accuracy {acc_f:.3f}  (fraction of responders {y_ring.mean():.2f})")
axes[0].legend(fontsize=7, loc="upper right"); plt.tight_layout(); plt.show()

🔍 **What just happened?** With the two raw signals, the best straight line is useless: it essentially predicts "no
response" everywhere and its accuracy is about the fraction of non-responders. Adding the two squared signals lets
the *same* algorithm draw a closed, roughly circular boundary and classify about 92 % of the cells correctly
(the rest is mostly the 5 % label noise plus the boundary region).

Hand-crafting features works when you know what shape to expect. The next modules learn non-linear boundaries
automatically: **decision trees and random forests** cut feature space into boxes ([Module 10](10_decision_trees_random_forests.ipynb)),
**support vector machines** with kernels implicitly use enormous sets of non-linear features ([Module 11](11_support_vector_machines.ipynb)),
and **neural networks** stack many logistic-regression-like units so that the features themselves are learned
([Module 22](22_neural_networks.ipynb)) — the last layer of a classification network *is* a logistic (softmax)
regression.

✏️ **Your turn.** Remove the noise (`flip = rng.random(400) < 0.0`) and set `lam=0.0` in the quadratic model. Do the
training data become separable, and what happens to the size of the weights? Connect your answer to Section 6.

---
## 📌 Key takeaways

- **Logistic regression models $P(Y \mid X)$ directly:** $P(Y{=}1\mid X) = \sigma(w_0 + \sum_i w_iX_i)$ with the logistic
  function $\sigma(b) = 1/(1 + e^{-b})$ — a smooth, probabilistic version of the sign function (σ(0) = ½, σ(±∞) = 1/0).
  (The slides use the opposite sign convention on slides 8–19; it only flips all weights.)
- **The log-odds is linear**, $\ln\frac{P(Y=1\mid X)}{P(Y=0\mid X)} = w_0 + \sum_i w_iX_i$, so the decision boundary is a
  hyperplane and $e^{w_i}$ is an odds ratio (conditional on the other features, not causal). R classes need R − 1
  weight vectors (equivalently a softmax).
- **Training maximises the conditional log-likelihood**
  $l(W) = \sum_\ell [Y^\ell z^\ell - \ln(1 + e^{z^\ell})]$, whose gradient is
  $\partial l/\partial w_i = \sum_\ell X_i^\ell (Y^\ell - \hat P(Y^\ell{=}1\mid X^\ell))$ — features × prediction errors.
- **No closed form, but concave:** gradient ascent $w \leftarrow w + \eta\nabla l$ reaches the global maximum if η is
  neither too small (slow) nor too large (overshoot/oscillation).
- **MAP with a Gaussian prior** $W \sim \mathcal N(0, \sigma^2 I)$ subtracts $\lambda\sum_i w_i^2$, $\lambda = 1/(2\sigma^2)$
  (weight decay, ridge). On **separable** data the MLE does not exist (weights → ∞, over-confident probabilities);
  the prior fixes it.
- **Real data:** ten training-selected Golub genes classify 33/34 independent test patients (precision 1.0, recall
  0.93) with honest, moderate probabilities.
- **Generative vs discriminative:** naïve Bayes models $P(X\mid Y)P(Y)$ (closed form; with shared Gaussian variances it
  has exactly the logistic form), logistic regression models $P(Y \mid X)$ (iterative). NB plateaus early and
  double-counts correlated evidence (poor calibration); LR has lower asymptotic error and better probabilities,
  especially when regularised.

---
## ✅ Checkpoint

1. Compute $P(Y=1 \mid x)$ for $w_0 = 2$, $w = (-1, 0.5)$ and $x = (3, 2)$. Which class is predicted, and what are the odds?
   <details><summary>Answer</summary>Score $z = 2 - 3 + 1 = 0$, so $P = \sigma(0) = 0.5$ and the odds are 1: the point lies exactly on the decision boundary (either class, depending on the tie rule).</details>
2. Starting from $l(W) = \sum_\ell [Y^\ell \ln P_1^\ell + (1 - Y^\ell)\ln P_0^\ell]$, derive $l(W) = \sum_\ell [Y^\ell z^\ell - \ln(1 + e^{z^\ell})]$.
   <details><summary>Answer</summary>Rewrite as $\sum_\ell [Y^\ell \ln(P_1^\ell/P_0^\ell) + \ln P_0^\ell]$. The model gives $P_1/P_0 = e^{z}$, so $\ln(P_1/P_0) = z$, and $P_0 = 1/(1 + e^z)$, so $\ln P_0 = -\ln(1 + e^z)$.</details>
3. Differentiate $\ln(1 + e^{z})$ with respect to $w_i$, where $z = w_0 + \sum_i w_ix_i$, and state the full gradient of $l(W)$.
   <details><summary>Answer</summary>$\frac{e^z}{1+e^z}\,x_i = \hat P(Y{=}1\mid x)\,x_i$ (chain rule with $(\ln f)' = f'/f$, $(e^f)' = e^f f'$). Hence $\partial l/\partial w_i = \sum_\ell x_i^\ell (y^\ell - \hat P(Y^\ell{=}1\mid x^\ell))$; in matrix form $X^\top(y - \hat p)$.</details>
4. Why does gradient ascent on the logistic-regression log-likelihood not get stuck in a poor local maximum, and what can still go wrong?
   <details><summary>Answer</summary>$l(W)$ is concave (second derivative $-\sum_\ell x_i^2\hat P(1-\hat P) \le 0$), so every local maximum is global. Things that can still go wrong: a step size that is too large (oscillation, Section 5.4) or too small (very slow), and separable data, where there is no maximum at all (Section 6).</details>
5. Your training data are perfectly separable. Describe what happens to $\lVert w\rVert$, to $l(W)$ and to the predicted training probabilities during unregularised gradient ascent, and how a prior $W\sim N(0,\sigma^2I)$ changes this.
   <details><summary>Answer</summary>$\lVert w \rVert$ grows without bound (logarithmically in the number of iterations), $l(W)$ increases towards 0 without reaching it, and the probabilities approach exactly 0 or 1. The prior adds $-\lambda\sum w_i^2$ with $\lambda = 1/(2\sigma^2)$, which makes the objective strictly concave with a finite maximiser: the weights converge.</details>
6. A test of 100 patients gives TP = 18, FP = 6, FN = 2, TN = 74. Compute accuracy, precision and recall.
   <details><summary>Answer</summary>Accuracy $= 92/100 = 0.92$; precision $= 18/24 = 0.75$; recall $= 18/20 = 0.90$.</details>
7. Show that Gaussian naïve Bayes with class-shared variances gives a linear log-odds, and give $w_i$.
   <details><summary>Answer</summary>$\ln\frac{\mathcal N(x_i;\mu_{i1},\sigma_i^2)}{\mathcal N(x_i;\mu_{i0},\sigma_i^2)} = \frac{(x_i-\mu_{i0})^2 - (x_i - \mu_{i1})^2}{2\sigma_i^2} = \frac{\mu_{i1}-\mu_{i0}}{\sigma_i^2}x_i + \frac{\mu_{i0}^2 - \mu_{i1}^2}{2\sigma_i^2}$; the $x_i^2$ terms cancel, so $w_i = (\mu_{i1} - \mu_{i0})/\sigma_i^2$.</details>
8. In code cell 5.5, what does the line `grad[1:] -= 2 * lam * w[1:]` implement, and why is `grad[0]` left alone?
   <details><summary>Answer</summary>The gradient of the log-prior $-\lambda\sum_{i\ge1} w_i^2$, i.e. $-2\lambda w_i$ (MAP / weight decay). The intercept $w_0$ (index 0) is not penalised, because shrinking it towards 0 would bias the base rate rather than control the complexity of the boundary.</details>

## 🧠 Questions & discussion

1. In a GWAS of a binary disease, each SNP is usually tested on its own with logistic regression, reporting an odds ratio. How does this differ from fitting one multivariable logistic regression with all SNPs, and why might the odds ratio of a SNP change between the two analyses?
2. Logistic regression's probabilities were better calibrated than naïve Bayes'. Would you expect them to stay calibrated if the test patients came from a different hospital, with a different proportion of AML? Which parameter would be most affected?
3. Slide 33's notes say that discriminative models "do not provide insights into the underlying data distribution". Give a biological task where you would prefer a generative model despite its worse calibration.
4. We chose the prior $\sigma^2 = 1$ (λ = 0.5) before looking at the test data. How would you choose λ in a principled way with only 38 training patients, and what is the danger of choosing it with the test set?
5. The ten Golub genes are strongly correlated, and logistic regression spread its weight over them. If a colleague concludes from the weight table that MYB is "less important" than cystatin C, what would you answer?

## 🚀 Bonus — going further

- **Deep dive (repository):** [`../../GenerativeDiscriminativeClassification/`](../../GenerativeDiscriminativeClassification/) implements Gaussian naïve Bayes and logistic regression from scratch on the Wisconsin breast-cancer data and studies decision boundaries and calibration (Brier score, expected calibration error, reliability diagrams) in detail — it complements Sections 8.1–8.2.
- **Regularisation paths:** the same ridge/lasso ideas for regression are developed on wheat genomics in [`../../RegularizedLinearRegression/`](../../RegularizedLinearRegression/); try an L1 penalty for logistic regression (`LogisticRegression(l1_ratio=1, solver='saga', C=...)` in scikit-learn ≥ 1.8; on older versions write `penalty='l1'` instead of `l1_ratio=1`) on all 3,571 Golub genes and see which genes survive.
- **Newton's method (IRLS).** Logistic regression is classically fitted by Newton's method: $w \leftarrow w + (X^\top S X)^{-1}X^\top(y - \hat p)$ with $S = \text{diag}(\hat p(1-\hat p))$. Implement it and compare the number of iterations with gradient ascent.
- **Prediction sets instead of probabilities.** Wrap the Golub classifier in conformal prediction ([`../../ConformalPrediction/`](../../ConformalPrediction/)) to obtain sets {ALL}, {AML} or {ALL, AML} with a guaranteed error rate.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            09 · Classification II — logistic regression, generative vs discriminative
Input:             simulated biomarker and protein data; iris (3 species); Golub et al. 1999 (38 train /
                   34 test, genes ranked on training patients); Wisconsin breast cancer (10 mean features)
Output:            l(W) surface, gradient-ascent learning curves, weight-norm growth on separable data,
                   Golub confusion matrix / precision / recall, NB vs LR learning curves and calibration
Method:            logistic regression by maximum conditional likelihood (gradient ascent), MAP with a
                   Gaussian prior (L2), comparison with Gaussian naïve Bayes
Key equation:      ∂l/∂w_i = Σ_ℓ x_i^ℓ (y^ℓ − P̂(Y^ℓ = 1 | x^ℓ));  λ = 1/(2σ²)
Key parameters:    step size η = ___ ; λ = ___ ; number of genes G = ___
What changed when I modified … (η / λ / G / duplicated features / squared features): ___
One failure mode:  ___ (e.g. separable data → weights diverge; η too large → oscillation)
Connection to 02-620 slides: 004_classification-LR slides ___
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, deck `004_classification-LR` (course design S. Kim; instructors M. Xu and M. Zhang).
- Mitchell T. M. (1997) *Machine Learning*. McGraw-Hill; and the companion chapter "Generative and discriminative classifiers: naive Bayes and logistic regression" (the source of the slides' notation).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 8 (logistic regression: model, gradient, Hessian, MAP/L2, multinomial) and §8.6 (generative vs discriminative).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §4.4 (logistic regression) and §4.3 (linear discriminant analysis, the shared-covariance generative cousin).
- Ng A. Y., Jordan M. I. (2002) On discriminative vs. generative classifiers: a comparison of logistic regression and naive Bayes. *Advances in Neural Information Processing Systems* 14.
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *Journal of the American Statistical Association* 97:77–87.
- Ambroise C., McLachlan G. J. (2002) Selection bias in gene extraction on the basis of microarray gene-expression data. *PNAS* 99:6562–6566.
- Fisher R. A. (1936) The use of multiple measurements in taxonomic problems. *Annals of Eugenics* 7:179–188 (iris data).
- Street W. N., Wolberg W. H., Mangasarian O. L. (1993) Nuclear feature extraction for breast tumor diagnosis. *Proc. SPIE* 1905:861–870 (Wisconsin diagnostic breast-cancer data).

## ➡️ Next

Logistic regression and naïve Bayes both describe classes with probability models. The next lecture takes a
completely different, non-probabilistic route — a flowchart of yes/no questions learned from data — and then
averages hundreds of such flowcharts to find cancer driver mutations:
➡️ [Module 10 · Decision trees, random forests and cancer driver mutations](10_decision_trees_random_forests.ipynb).